# Roll-rate de Collections — Jun → Jul → Ago e projeção de Setembro

In [3]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [9]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\beelt\Documents\collections_case_candidate")

print("Procurando CSVs dentro de:")
print(PROJECT_DIR)
print("=" * 80)

csv_files = list(PROJECT_DIR.rglob("*.csv"))

print(f"\n{len(csv_files)} CSV(s) encontrado(s):\n")

for file in csv_files:
    print(file.relative_to(PROJECT_DIR))

Procurando CSVs dentro de:
C:\Users\beelt\Documents\collections_case_candidate

271 CSV(s) encontrado(s):

data\interim\whatsapp_customer_canonical.csv
data\interim\whatsapp_customer_canonical_reconciliation.csv
data\interim\whatsapp_customer_feature_dictionary.csv
data\interim\whatsapp_interactions_enriched.csv
data\raw\collections_queue_sep2026.csv
data\raw\whatsapp_collections_history.csv
.venv\Lib\site-packages\tornado\test\csv_translations\fr_FR.csv
.venv\Lib\site-packages\statsmodels\tsa\vector_ar\tests\Matlab_results\test_coint.csv
.venv\Lib\site-packages\statsmodels\tsa\tests\results\arima111_forecasts.csv
.venv\Lib\site-packages\statsmodels\tsa\tests\results\arima212_forecast.csv
.venv\Lib\site-packages\statsmodels\tsa\tests\results\ARMLEConstantPredict.csv
.venv\Lib\site-packages\statsmodels\tsa\tests\results\AROLSConstantPredict.csv
.venv\Lib\site-packages\statsmodels\tsa\tests\results\AROLSNoConstantPredict.csv
.venv\Lib\site-packages\statsmodels\tsa\tests\results\BAA.csv
.

In [10]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\beelt\Documents\collections_case_candidate")

# ------------------------------------------------------------
# Find datasets automatically
# ------------------------------------------------------------

wa_candidates = list(
    PROJECT_DIR.rglob("*whatsapp_collections_history*.csv")
)

queue_candidates = list(
    PROJECT_DIR.rglob("*collections_queue_sep2026*.csv")
)

print("WhatsApp candidates:")
for f in wa_candidates:
    print(" -", f.relative_to(PROJECT_DIR))

print("\nQueue candidates:")
for f in queue_candidates:
    print(" -", f.relative_to(PROJECT_DIR))

WhatsApp candidates:
 - data\raw\whatsapp_collections_history.csv

Queue candidates:
 - data\raw\collections_queue_sep2026.csv


In [11]:
if len(wa_candidates) != 1:
    raise ValueError(
        f"Expected 1 WhatsApp file, found {len(wa_candidates)}"
    )

if len(queue_candidates) != 1:
    raise ValueError(
        f"Expected 1 Queue file, found {len(queue_candidates)}"
    )

WA_PATH = wa_candidates[0]
QUEUE_PATH = queue_candidates[0]

wa = pd.read_csv(WA_PATH)
queue = pd.read_csv(QUEUE_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"])

if "in_collections_since" in queue.columns:
    queue["in_collections_since"] = pd.to_datetime(
        queue["in_collections_since"]
    )

print("\n" + "=" * 70)
print("DATA LOADED SUCCESSFULLY")
print("=" * 70)

print(
    f"WhatsApp : {len(wa):,} rows | "
    f"{wa['customer_id'].nunique():,} customers"
)

print(
    f"Queue Sep: {len(queue):,} rows | "
    f"{queue['customer_id'].nunique():,} customers"
)

print(
    f"WA period: {wa['sent_at'].min()} → "
    f"{wa['sent_at'].max()}"
)


DATA LOADED SUCCESSFULLY
WhatsApp : 75,406 rows | 11,724 customers
Queue Sep: 10,658 rows | 10,658 customers
WA period: 2026-06-01 09:18:00 → 2026-08-31 20:59:00


## 1. Reconstrução temporal da dívida

Como não temos a data de vencimento explicitamente, calculamos para cada mensagem:

`due_date_implícita = data_do_envio - DPD`

Depois usamos a **mediana da due date implícita por cliente**, que é robusta a pequenas inconsistências entre linhas.

In [12]:
wa["sent_date"] = wa["sent_at"].dt.normalize()
wa["implied_due_date"] = wa["sent_date"] - pd.to_timedelta(wa["days_past_due"], unit="D")

due = (
    wa.groupby("customer_id", as_index=False)
      .agg(
          implied_due_date=("implied_due_date", "median"),
          first_seen=("sent_at", "min"),
          last_seen=("sent_at", "max")
      )
)

# dispersão da due date implícita: auditoria de consistência
audit_due = (
    wa.groupby("customer_id")["implied_due_date"]
      .agg(["min","max"])
      .assign(spread_days=lambda x: (x["max"] - x["min"]).dt.days)
)

print(audit_due["spread_days"].describe(percentiles=[.5,.75,.9,.95,.99]))

count   11,724.00
mean         0.00
std          0.00
min          0.00
50%          0.00
75%          0.00
90%          0.00
95%          0.00
99%          0.00
max          0.00
Name: spread_days, dtype: float64


## 2. Proxy de quitação

Consideramos um evento como quitação quando:
- template normal: `amount_paid_brl >= outstanding_balance_brl`; ou
- `discount_offer`: o pagamento atinge **85% do saldo**, conforme regra do case.

Como só temos `paid_within_72h` e não o timestamp exato do pagamento, usamos `sent_at` como **proxy conservadora do momento em que a quitação foi observada**.

In [13]:
wa["amount_paid_brl"] = wa["amount_paid_brl"].fillna(0)
wa["paid_within_72h"] = wa["paid_within_72h"].fillna(0).astype(int)

wa["settlement_threshold"] = np.where(
    wa["template"].eq("discount_offer"),
    0.85 * wa["outstanding_balance_brl"],
    wa["outstanding_balance_brl"]
)

wa["full_payment_event"] = (
    wa["paid_within_72h"].eq(1)
    & (wa["amount_paid_brl"] >= wa["settlement_threshold"] - 0.01)
)

settled = (
    wa.loc[wa["full_payment_event"]]
      .groupby("customer_id", as_index=False)
      .agg(settled_at=("sent_at", "min"))
)

customer = due.merge(settled, on="customer_id", how="left")

print("Clientes com proxy de quitação:", customer["settled_at"].notna().sum())

Clientes com proxy de quitação: 3641


## 3. Função de estado no snapshot

A matriz é construída em snapshots mensais. Para evitar *look-ahead*, um cliente só entra em um snapshot se já era observável na base até aquela data.

`60d+` é tratado como estado terminal para a régua de WhatsApp deste case. `Quitado` também é absorvente.

In [14]:
BUCKETS = ["1-15d", "16-30d", "31-45d", "46-59d", "60d+", "Quitado"]

def dpd_bucket(dpd):
    if pd.isna(dpd) or dpd < 1:
        return np.nan
    if dpd <= 15:
        return "1-15d"
    if dpd <= 30:
        return "16-30d"
    if dpd <= 45:
        return "31-45d"
    if dpd <= 59:
        return "46-59d"
    return "60d+"

def snapshot_states(customer_df, snapshot_date):
    s = pd.Timestamp(snapshot_date).normalize()
    x = customer_df.copy()

    # conhecido/observável até o snapshot
    x = x[x["first_seen"].dt.normalize() <= s].copy()

    # Quitado é absorvente após a primeira quitação observada
    is_settled = x["settled_at"].notna() & (x["settled_at"].dt.normalize() <= s)

    x["dpd_snapshot"] = (s - x["implied_due_date"].dt.normalize()).dt.days
    x["state"] = x["dpd_snapshot"].map(dpd_bucket)
    x.loc[is_settled, "state"] = "Quitado"

    # ainda não inadimplente no snapshot não entra na matriz
    x = x[x["state"].notna()].copy()
    return x[["customer_id","state","dpd_snapshot"]]

snap_jun = snapshot_states(customer, "2026-06-01")
snap_jul = snapshot_states(customer, "2026-07-01")
snap_aug = snapshot_states(customer, "2026-08-01")

print("Snapshot Jun:", len(snap_jun))
print("Snapshot Jul:", len(snap_jul))
print("Snapshot Ago:", len(snap_aug))

Snapshot Jun: 46
Snapshot Jul: 3830
Snapshot Ago: 7906


In [15]:
# ============================================================
# AUDIT — CUSTOMER COVERAGE BY MONTH
# ============================================================

wa["month"] = wa["sent_at"].dt.to_period("M")

coverage = (
    wa.groupby("month")
      .agg(
          messages=("customer_id", "size"),
          customers=("customer_id", "nunique"),
          min_date=("sent_at", "min"),
          max_date=("sent_at", "max")
      )
)

display(coverage)

print("\nCustomers observed in both Jun and Jul:")
jun_ids = set(
    wa.loc[wa["month"] == "2026-06", "customer_id"]
)
jul_ids = set(
    wa.loc[wa["month"] == "2026-07", "customer_id"]
)
aug_ids = set(
    wa.loc[wa["month"] == "2026-08", "customer_id"]
)

print(f"Jun customers      : {len(jun_ids):,}")
print(f"Jul customers      : {len(jul_ids):,}")
print(f"Aug customers      : {len(aug_ids):,}")

print(f"\nJun ∩ Jul          : {len(jun_ids & jul_ids):,}")
print(f"Jul ∩ Aug          : {len(jul_ids & aug_ids):,}")
print(f"Jun ∩ Jul ∩ Aug    : {len(jun_ids & jul_ids & aug_ids):,}")

,messages,customers,min_date,max_date
month,,,,
2026-06,14465,3681,2026-06-01 09:18:00,2026-06-30 20:59:00
2026-07,29716,7030,2026-07-01 09:00:00,2026-07-31 20:59:00
2026-08,31225,8595,2026-08-01 09:00:00,2026-08-31 20:59:00



Customers observed in both Jun and Jul:
Jun customers      : 3,681
Jul customers      : 7,030
Aug customers      : 8,595

Jun ∩ Jul          : 2,875
Jul ∩ Aug          : 4,688
Jun ∩ Jul ∩ Aug    : 1,461


In [17]:
# ============================================================
# 3. FORMAL MONTH-END ROLL-RATE
# ============================================================

BUCKETS = [
    "1-15d",
    "16-30d",
    "31-45d",
    "46-59d",
    "60d+",
    "Quitado"
]


def dpd_bucket(dpd):

    if pd.isna(dpd) or dpd < 1:
        return np.nan

    if dpd <= 15:
        return "1-15d"

    if dpd <= 30:
        return "16-30d"

    if dpd <= 45:
        return "31-45d"

    if dpd <= 59:
        return "46-59d"

    return "60d+"


# ============================================================
# MONTH-END STATE
# ============================================================

def month_end_state(customer_df, snapshot_date):

    snapshot = pd.Timestamp(snapshot_date).normalize()

    x = customer_df.copy()

    # --------------------------------------------------------
    # DPD implied by due date
    # --------------------------------------------------------

    x["dpd_snapshot"] = (
        snapshot
        - x["implied_due_date"].dt.normalize()
    ).dt.days

    x["state"] = (
        x["dpd_snapshot"]
        .apply(dpd_bucket)
    )

    # --------------------------------------------------------
    # Settlement overrides DPD
    # --------------------------------------------------------

    settled_before_snapshot = (
        x["settled_at"].notna()
        &
        (
            x["settled_at"].dt.normalize()
            <= snapshot
        )
    )

    x.loc[
        settled_before_snapshot,
        "state"
    ] = "Quitado"

    return x[
        [
            "customer_id",
            "dpd_snapshot",
            "state",
            "settled_at",
            "first_seen",
            "last_seen"
        ]
    ]

In [18]:
# ============================================================
# 4. MONTH-END SNAPSHOTS
# ============================================================

state_jun30 = month_end_state(
    customer,
    "2026-06-30"
)

state_jul31 = month_end_state(
    customer,
    "2026-07-31"
)

state_aug31 = month_end_state(
    customer,
    "2026-08-31"
)

In [19]:
# ============================================================
# 5. POINT-IN-TIME ELIGIBLE POPULATIONS
# ============================================================

jun_cohort = customer.loc[
    customer["first_seen"].dt.normalize()
    <= pd.Timestamp("2026-06-30")
]["customer_id"]

jul_cohort = customer.loc[
    customer["first_seen"].dt.normalize()
    <= pd.Timestamp("2026-07-31")
]["customer_id"]


jun30 = state_jun30[
    state_jun30["customer_id"].isin(jun_cohort)
].copy()

jul31_for_jun = state_jul31[
    state_jul31["customer_id"].isin(jun_cohort)
].copy()


jul31 = state_jul31[
    state_jul31["customer_id"].isin(jul_cohort)
].copy()

aug31_for_jul = state_aug31[
    state_aug31["customer_id"].isin(jul_cohort)
].copy()


print("=" * 70)
print("ELIGIBLE POPULATIONS")
print("=" * 70)

print(
    f"Jun → Jul cohort : "
    f"{jun30['customer_id'].nunique():,}"
)

print(
    f"Jul → Aug cohort : "
    f"{jul31['customer_id'].nunique():,}"
)

ELIGIBLE POPULATIONS
Jun → Jul cohort : 3,681
Jul → Aug cohort : 7,836


In [20]:
# ============================================================
# 6. ROLL-RATE FUNCTION
# ============================================================

def roll_matrix(
    state_from,
    state_to
):

    trans = (
        state_from[
            ["customer_id", "state"]
        ]
        .rename(
            columns={
                "state": "from_state"
            }
        )
        .merge(
            state_to[
                ["customer_id", "state"]
            ]
            .rename(
                columns={
                    "state": "to_state"
                }
            ),
            on="customer_id",
            how="inner"
        )
    )

    # Remove estado não aplicável
    trans = trans[
        trans["from_state"].notna()
        &
        trans["to_state"].notna()
    ].copy()

    counts = (
        pd.crosstab(
            trans["from_state"],
            trans["to_state"]
        )
        .reindex(
            index=BUCKETS,
            columns=BUCKETS,
            fill_value=0
        )
    )

    pct = (
        counts
        .div(
            counts.sum(axis=1)
            .replace(0, np.nan),
            axis=0
        )
        * 100
    )

    pct["n"] = counts.sum(axis=1)

    return trans, counts, pct

In [21]:
# ============================================================
# 7. JUN → JUL
# ============================================================

trans_jun_jul, count_jun_jul, pct_jun_jul = (
    roll_matrix(
        jun30,
        jul31_for_jun
    )
)


print("=" * 90)
print("ROLL-RATE | JUN 30 → JUL 31")
print("=" * 90)

display(
    pct_jun_jul.round(1)
)

ROLL-RATE | JUN 30 → JUL 31


to_state,1-15d,16-30d,31-45d,46-59d,60d+,Quitado,n
from_state,,,,,,,
1-15d,0.00,0.00,71.90,6.00,0.00,22.10,1441
16-30d,0.00,0.00,0.00,75.40,10.00,14.60,1477
31-45d,NaN,NaN,NaN,NaN,NaN,NaN,0
46-59d,NaN,NaN,NaN,NaN,NaN,NaN,0
60d+,NaN,NaN,NaN,NaN,NaN,NaN,0
Quitado,0.00,0.00,0.00,0.00,0.00,100.00,763


In [22]:
# ============================================================
# JUL → AUG
# ============================================================

trans_jul_aug, count_jul_aug, pct_jul_aug = (
    roll_matrix(
        jul31,
        aug31_for_jul
    )
)

print("=" * 90)
print("ROLL-RATE | JUL 31 → AUG 31")
print("=" * 90)

display(
    pct_jul_aug.round(1)
)

ROLL-RATE | JUL 31 → AUG 31


to_state,1-15d,16-30d,31-45d,46-59d,60d+,Quitado,n
from_state,,,,,,,
1-15d,0.00,0.00,73.30,6.20,0.00,20.50,1474
16-30d,0.00,0.00,0.00,76.30,10.30,13.40,1520
31-45d,0.00,0.00,0.00,0.00,92.50,7.50,1318
46-59d,0.00,0.00,0.00,0.00,96.70,3.30,1202
60d+,0.00,0.00,0.00,0.00,100.00,0.00,147
Quitado,0.00,0.00,0.00,0.00,0.00,100.00,2175


In [23]:
# ============================================================
# BUSINESS VIEW — CURE VS DETERIORATION
# ============================================================

def business_roll_view(pct_matrix):

    rows = []

    for bucket in [
        "1-15d",
        "16-30d",
        "31-45d",
        "46-59d",
        "60d+"
    ]:

        if bucket not in pct_matrix.index:
            continue

        n = pct_matrix.loc[bucket, "n"]

        if n == 0:
            continue

        cure = pct_matrix.loc[
            bucket,
            "Quitado"
        ]

        to_60 = pct_matrix.loc[
            bucket,
            "60d+"
        ]

        rows.append({
            "bucket": bucket,
            "customers": int(n),
            "cure_rate_pct": cure,
            "roll_to_60plus_pct": to_60,
            "remain_unsettled_pct": 100 - cure
        })

    return pd.DataFrame(rows)


print("JUN → JUL")
display(
    business_roll_view(
        pct_jun_jul
    ).round(1)
)

print("\nJUL → AUG")
display(
    business_roll_view(
        pct_jul_aug
    ).round(1)
)

JUN → JUL


,bucket,customers,cure_rate_pct,roll_to_60plus_pct,remain_unsettled_pct
0,1-15d,1441,22.10,0.00,77.90
1,16-30d,1477,14.60,10.00,85.40



JUL → AUG


,bucket,customers,cure_rate_pct,roll_to_60plus_pct,remain_unsettled_pct
0,1-15d,1474,20.50,0.00,79.50
1,16-30d,1520,13.40,10.30,86.60
2,31-45d,1318,7.50,92.50,92.50
3,46-59d,1202,3.30,96.70,96.70
4,60d+,147,0.00,100.00,100.00


In [24]:
# ============================================================
# 9. SEPTEMBER PORTFOLIO — REAL SNAPSHOT
# ============================================================

# Detect DPD column
dpd_candidates = [
    c for c in queue.columns
    if "past_due" in c.lower() or "dpd" in c.lower()
]

print("Possible DPD columns:")
print(dpd_candidates)

print("\nQueue columns:")
print(queue.columns.tolist())

Possible DPD columns:
['days_past_due_on_2026-09-01']

Queue columns:
['customer_id', 'in_collections_since', 'days_past_due_on_2026-09-01', 'outstanding_balance_brl', 'monthly_salary_brl', 'payday_day_of_month', 'n_prior_transactions', 'account_age_months', 'days_since_last_app_login', 'state_uf']


In [25]:
# ============================================================
# SEPTEMBER BUCKET
# ============================================================

DPD_COL = "days_past_due_on_2026-09-01"

queue["sep_bucket"] = (
    queue[DPD_COL]
    .apply(dpd_bucket)
)

sep_portfolio = (
    queue
    .groupby(
        "sep_bucket",
        observed=False
    )
    .agg(
        customers=(
            "customer_id",
            "nunique"
        ),
        balance_brl=(
            "outstanding_balance_brl",
            "sum"
        ),
        avg_balance_brl=(
            "outstanding_balance_brl",
            "mean"
        )
    )
    .reindex(
        [
            "1-15d",
            "16-30d",
            "31-45d",
            "46-59d",
            "60d+"
        ]
    )
    .fillna(0)
)

sep_portfolio["customer_share_pct"] = (
    sep_portfolio["customers"]
    / sep_portfolio["customers"].sum()
    * 100
)

sep_portfolio["balance_share_pct"] = (
    sep_portfolio["balance_brl"]
    / sep_portfolio["balance_brl"].sum()
    * 100
)

display(
    sep_portfolio.round(2)
)

print("=" * 70)
print("SEPTEMBER PORTFOLIO")
print("=" * 70)

print(
    f"Customers : "
    f"{sep_portfolio['customers'].sum():,.0f}"
)

print(
    f"Balance   : "
    f"R$ {sep_portfolio['balance_brl'].sum():,.2f}"
)

,customers,balance_brl,avg_balance_brl,customer_share_pct,balance_share_pct
sep_bucket,,,,,
1-15d,1559,"1,280,571.56",821.41,27.55,27.83
16-30d,1432,"1,192,227.01",832.56,25.31,25.91
31-45d,1340,"1,090,654.81",813.92,23.68,23.70
46-59d,1243,"971,487.73",781.57,21.97,21.11
60d+,84,"66,708.45",794.15,1.48,1.45


SEPTEMBER PORTFOLIO
Customers : 5,658
Balance   : R$ 4,601,649.56


In [26]:
# ============================================================
# 10. SEPTEMBER EXISTING PORTFOLIO × HISTORICAL ROLL-RATE
# ============================================================

# Jul → Aug observed cure rates
historical_cure = {
    "1-15d": 0.205,
    "16-30d": 0.134,
    "31-45d": 0.075,
    "46-59d": 0.033,
    "60d+": 0.000
}

historical_roll_60 = {
    "1-15d": 0.000,
    "16-30d": 0.103,
    "31-45d": 0.925,
    "46-59d": 0.967,
    "60d+": 1.000
}


projection = sep_portfolio.copy()

projection["cure_rate"] = (
    projection.index
    .map(historical_cure)
)

projection["roll_60plus_rate"] = (
    projection.index
    .map(historical_roll_60)
)


# ------------------------------------------------------------
# EXPECTED CUSTOMER FLOW
# ------------------------------------------------------------

projection["expected_cured_customers"] = (
    projection["customers"]
    * projection["cure_rate"]
)

projection["expected_unsettled_customers"] = (
    projection["customers"]
    * (1 - projection["cure_rate"])
)

projection["expected_60plus_customers"] = (
    projection["customers"]
    * projection["roll_60plus_rate"]
)


# ------------------------------------------------------------
# BALANCE EXPOSURE
#
# Important:
# These are balance exposures associated with historical rates.
# They are NOT accounting expected loss.
# ------------------------------------------------------------

projection["balance_associated_cure"] = (
    projection["balance_brl"]
    * projection["cure_rate"]
)

projection["balance_remaining_unsettled"] = (
    projection["balance_brl"]
    * (1 - projection["cure_rate"])
)

projection["balance_associated_60plus"] = (
    projection["balance_brl"]
    * projection["roll_60plus_rate"]
)


# ------------------------------------------------------------
# PRESENTATION VIEW
# ------------------------------------------------------------

projection_view = projection[
    [
        "customers",
        "balance_brl",
        "cure_rate",
        "expected_cured_customers",
        "balance_associated_cure",
        "expected_60plus_customers",
        "balance_associated_60plus"
    ]
].copy()


projection_view["cure_rate"] *= 100

projection_view = projection_view.rename(
    columns={
        "customers": "customers_sep",
        "balance_brl": "balance_sep_brl",
        "cure_rate": "historical_cure_pct",
        "expected_cured_customers": "projected_cured_customers",
        "balance_associated_cure": "balance_associated_cure_brl",
        "expected_60plus_customers": "projected_60plus_customers",
        "balance_associated_60plus": "balance_associated_60plus_brl"
    }
)


print("=" * 100)
print("SEPTEMBER EXISTING PORTFOLIO — ROLL-RATE SCENARIO")
print("=" * 100)

display(
    projection_view.round(2)
)

SEPTEMBER EXISTING PORTFOLIO — ROLL-RATE SCENARIO


,customers_sep,balance_sep_brl,historical_cure_pct,projected_cured_customers,balance_associated_cure_brl,projected_60plus_customers,balance_associated_60plus_brl
sep_bucket,,,,,,,
1-15d,1559,"1,280,571.56",20.50,319.59,"262,517.17",0.00,0.00
16-30d,1432,"1,192,227.01",13.40,191.89,"159,758.42",147.50,"122,799.38"
31-45d,1340,"1,090,654.81",7.50,100.50,"81,799.11","1,239.50","1,008,855.70"
46-59d,1243,"971,487.73",3.30,41.02,"32,059.10","1,201.98","939,428.63"
60d+,84,"66,708.45",0.00,0.00,0.00,84.00,"66,708.45"


In [27]:
# ============================================================
# 11. EXECUTIVE SUMMARY
# ============================================================

total_cured = (
    projection["expected_cured_customers"]
    .sum()
)

total_balance_cure = (
    projection["balance_associated_cure"]
    .sum()
)

total_60plus = (
    projection["expected_60plus_customers"]
    .sum()
)

total_balance_60plus = (
    projection["balance_associated_60plus"]
    .sum()
)


print("=" * 80)
print("ROLL-RATE SCENARIO — SEPTEMBER PRE-EXISTING PORTFOLIO")
print("=" * 80)

print(
    f"Portfolio               : "
    f"{projection['customers'].sum():,.0f} customers"
)

print(
    f"Outstanding balance     : "
    f"R$ {projection['balance_brl'].sum():,.2f}"
)

print()

print(
    f"Projected cures         : "
    f"{total_cured:,.0f} customers"
)

print(
    f"Balance associated cure : "
    f"R$ {total_balance_cure:,.2f}"
)

print()

print(
    f"Projected 60d+          : "
    f"{total_60plus:,.0f} customers"
)

print(
    f"Balance associated 60d+ : "
    f"R$ {total_balance_60plus:,.2f}"
)

ROLL-RATE SCENARIO — SEPTEMBER PRE-EXISTING PORTFOLIO
Portfolio               : 5,658 customers
Outstanding balance     : R$ 4,601,649.56

Projected cures         : 653 customers
Balance associated cure : R$ 536,133.79

Projected 60d+          : 2,673 customers
Balance associated 60d+ : R$ 2,137,792.17


In [28]:
# ============================================================
# 12. BALANCE-WEIGHTED RECOVERY BY ORIGIN DPD
# ============================================================

import pandas as pd
import numpy as np


BUCKET_ORDER = [
    "1-15d",
    "16-30d",
    "31-45d",
    "46-59d",
    "60d+"
]


def dpd_bucket(dpd):

    if pd.isna(dpd) or dpd < 1:
        return np.nan

    if dpd <= 15:
        return "1-15d"

    if dpd <= 30:
        return "16-30d"

    if dpd <= 45:
        return "31-45d"

    if dpd <= 59:
        return "46-59d"

    return "60d+"


def monetary_recovery_window(
    wa,
    customer,
    snapshot_start,
    snapshot_end
):

    start = pd.Timestamp(snapshot_start)
    end = pd.Timestamp(snapshot_end)

    # --------------------------------------------------------
    # 1. Customers known by beginning of window
    # --------------------------------------------------------

    eligible = customer.loc[
        customer["first_seen"].dt.normalize() <= start,
        [
            "customer_id",
            "implied_due_date",
            "settled_at"
        ]
    ].copy()

    # --------------------------------------------------------
    # 2. Remove customers already settled before start
    # --------------------------------------------------------

    eligible = eligible[
        eligible["settled_at"].isna()
        |
        (
            eligible["settled_at"].dt.normalize()
            > start
        )
    ].copy()

    # --------------------------------------------------------
    # 3. DPD at beginning of window
    # --------------------------------------------------------

    eligible["dpd_start"] = (
        start
        - eligible["implied_due_date"].dt.normalize()
    ).dt.days

    eligible["origin_bucket"] = (
        eligible["dpd_start"]
        .apply(dpd_bucket)
    )

    eligible = eligible[
        eligible["origin_bucket"].notna()
    ].copy()

    # --------------------------------------------------------
    # 4. Get last observed balance AT OR BEFORE snapshot
    #
    # PIT rule:
    # never use a balance observed after the snapshot
    # --------------------------------------------------------

    hist_before = wa[
        wa["sent_at"].dt.normalize() <= start
    ].copy()

    hist_before = (
        hist_before
        .sort_values(
            ["customer_id", "sent_at"]
        )
        .groupby(
            "customer_id",
            as_index=False
        )
        .tail(1)
    )

    balance_start = hist_before[
        [
            "customer_id",
            "sent_at",
            "outstanding_balance_brl"
        ]
    ].rename(
        columns={
            "sent_at": "balance_observed_at",
            "outstanding_balance_brl": "balance_start_brl"
        }
    )

    eligible = eligible.merge(
        balance_start,
        on="customer_id",
        how="left"
    )

    # --------------------------------------------------------
    # 5. Payments strictly AFTER start and up to end
    # --------------------------------------------------------

    payments = wa[
        (wa["sent_at"].dt.normalize() > start)
        &
        (wa["sent_at"].dt.normalize() <= end)
        &
        (wa["paid_within_72h"].eq(1))
        &
        (wa["amount_paid_brl"] > 0)
    ].copy()

    payments_by_customer = (
        payments
        .groupby(
            "customer_id",
            as_index=False
        )
        .agg(
            recovered_brl=(
                "amount_paid_brl",
                "sum"
            ),
            payment_events=(
                "amount_paid_brl",
                "size"
            )
        )
    )

    eligible = eligible.merge(
        payments_by_customer,
        on="customer_id",
        how="left"
    )

    eligible["recovered_brl"] = (
        eligible["recovered_brl"]
        .fillna(0)
    )

    eligible["payment_events"] = (
        eligible["payment_events"]
        .fillna(0)
        .astype(int)
    )

    # --------------------------------------------------------
    # 6. Sanity cap
    #
    # Recovery cannot exceed starting exposure.
    # Keep raw amount for auditing.
    # --------------------------------------------------------

    eligible["recovered_raw_brl"] = (
        eligible["recovered_brl"]
    )

    eligible["recovered_brl"] = np.minimum(
        eligible["recovered_brl"],
        eligible["balance_start_brl"]
    )

    # --------------------------------------------------------
    # 7. Recovery rate at customer level
    # --------------------------------------------------------

    eligible["recovery_rate_customer"] = np.where(
        eligible["balance_start_brl"] > 0,
        eligible["recovered_brl"]
        / eligible["balance_start_brl"],
        np.nan
    )

    # --------------------------------------------------------
    # 8. Aggregate by origin DPD
    # --------------------------------------------------------

    result = (
        eligible
        .groupby(
            "origin_bucket",
            observed=False
        )
        .agg(
            customers=(
                "customer_id",
                "nunique"
            ),

            customers_with_balance=(
                "balance_start_brl",
                "count"
            ),

            starting_balance_brl=(
                "balance_start_brl",
                "sum"
            ),

            recovered_brl=(
                "recovered_brl",
                "sum"
            ),

            customers_with_payment=(
                "payment_events",
                lambda x: (x > 0).sum()
            ),

            avg_starting_balance_brl=(
                "balance_start_brl",
                "mean"
            )
        )
        .reindex(BUCKET_ORDER)
    )

    # --------------------------------------------------------
    # 9. Balance-weighted recovery rate
    # --------------------------------------------------------

    result["balance_recovery_rate_pct"] = (
        result["recovered_brl"]
        / result["starting_balance_brl"]
        * 100
    )

    result["customer_payment_rate_pct"] = (
        result["customers_with_payment"]
        / result["customers"]
        * 100
    )

    result["balance_coverage_pct"] = (
        result["customers_with_balance"]
        / result["customers"]
        * 100
    )

    return eligible, result

In [29]:
# ============================================================
# JUN 30 → JUL 31
# ============================================================

money_jun_jul_detail, money_jun_jul = (
    monetary_recovery_window(
        wa=wa,
        customer=customer,
        snapshot_start="2026-06-30",
        snapshot_end="2026-07-31"
    )
)

print("=" * 100)
print("BALANCE RECOVERY | JUN 30 → JUL 31")
print("=" * 100)

display(
    money_jun_jul.round(2)
)

BALANCE RECOVERY | JUN 30 → JUL 31


,customers,customers_with_balance,starting_balance_brl,recovered_brl,customers_with_payment,avg_starting_balance_brl,balance_recovery_rate_pct,customer_payment_rate_pct,balance_coverage_pct
origin_bucket,,,,,,,,,
1-15d,"1,441.00","1,441.00","1,202,669.65","291,435.97",441.00,834.61,24.23,30.60,100.00
16-30d,"1,477.00","1,477.00","1,196,708.79","187,879.14",308.00,810.23,15.70,20.85,100.00
31-45d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46-59d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60d+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# ============================================================
# JUL 31 → AUG 31
# ============================================================

money_jul_aug_detail, money_jul_aug = (
    monetary_recovery_window(
        wa=wa,
        customer=customer,
        snapshot_start="2026-07-31",
        snapshot_end="2026-08-31"
    )
)

print("=" * 100)
print("BALANCE RECOVERY | JUL 31 → AUG 31")
print("=" * 100)

display(
    money_jul_aug.round(2)
)

BALANCE RECOVERY | JUL 31 → AUG 31


,customers,customers_with_balance,starting_balance_brl,recovered_brl,customers_with_payment,avg_starting_balance_brl,balance_recovery_rate_pct,customer_payment_rate_pct,balance_coverage_pct
origin_bucket,,,,,,,,,
1-15d,1474,1474,"1,270,980.67","278,270.22",439,862.27,21.89,29.78,100.00
16-30d,1520,1520,"1,215,472.82","159,862.09",275,799.65,13.15,18.09,100.00
31-45d,1318,1318,"1,026,239.53","76,241.72",140,778.63,7.43,10.62,100.00
46-59d,1202,1202,"957,404.43","22,894.38",44,796.51,2.39,3.66,100.00
60d+,147,147,"111,428.15",0.00,0,758.01,0.00,0.00,100.00


In [31]:
# ============================================================
# 13. CUSTOMER CURE × BALANCE RECOVERY
# ============================================================

cure_jul_aug = (
    business_roll_view(
        pct_jul_aug
    )
    .set_index("bucket")
)

money_comparison = (
    money_jul_aug[
        [
            "customers",
            "starting_balance_brl",
            "recovered_brl",
            "balance_recovery_rate_pct",
            "customer_payment_rate_pct",
            "balance_coverage_pct"
        ]
    ]
    .copy()
)

money_comparison["cure_rate_pct"] = (
    cure_jul_aug["cure_rate_pct"]
)

money_comparison["roll_to_60plus_pct"] = (
    cure_jul_aug["roll_to_60plus_pct"]
)


money_comparison = money_comparison[
    [
        "customers",
        "starting_balance_brl",
        "cure_rate_pct",
        "customer_payment_rate_pct",
        "recovered_brl",
        "balance_recovery_rate_pct",
        "roll_to_60plus_pct",
        "balance_coverage_pct"
    ]
]


print("=" * 110)
print("CUSTOMER CURE × MONETARY RECOVERY | JUL → AUG")
print("=" * 110)

display(
    money_comparison.round(2)
)

CUSTOMER CURE × MONETARY RECOVERY | JUL → AUG


,customers,starting_balance_brl,cure_rate_pct,customer_payment_rate_pct,recovered_brl,balance_recovery_rate_pct,roll_to_60plus_pct,balance_coverage_pct
origin_bucket,,,,,,,,
1-15d,1474,"1,270,980.67",20.49,29.78,"278,270.22",21.89,0.00,100.00
16-30d,1520,"1,215,472.82",13.36,18.09,"159,862.09",13.15,10.33,100.00
31-45d,1318,"1,026,239.53",7.51,10.62,"76,241.72",7.43,92.49,100.00
46-59d,1202,"957,404.43",3.33,3.66,"22,894.38",2.39,96.67,100.00
60d+,147,"111,428.15",0.00,0.00,0.00,0.00,100.00,100.00


In [32]:
# ============================================================
# 13. CUSTOMER CURE × BALANCE RECOVERY
# ============================================================

cure_jul_aug = (
    business_roll_view(
        pct_jul_aug
    )
    .set_index("bucket")
)

money_comparison = (
    money_jul_aug[
        [
            "customers",
            "starting_balance_brl",
            "recovered_brl",
            "balance_recovery_rate_pct",
            "customer_payment_rate_pct",
            "balance_coverage_pct"
        ]
    ]
    .copy()
)

money_comparison["cure_rate_pct"] = (
    cure_jul_aug["cure_rate_pct"]
)

money_comparison["roll_to_60plus_pct"] = (
    cure_jul_aug["roll_to_60plus_pct"]
)


money_comparison = money_comparison[
    [
        "customers",
        "starting_balance_brl",
        "cure_rate_pct",
        "customer_payment_rate_pct",
        "recovered_brl",
        "balance_recovery_rate_pct",
        "roll_to_60plus_pct",
        "balance_coverage_pct"
    ]
]


print("=" * 110)
print("CUSTOMER CURE × MONETARY RECOVERY | JUL → AUG")
print("=" * 110)

display(
    money_comparison.round(2)
)

CUSTOMER CURE × MONETARY RECOVERY | JUL → AUG


,customers,starting_balance_brl,cure_rate_pct,customer_payment_rate_pct,recovered_brl,balance_recovery_rate_pct,roll_to_60plus_pct,balance_coverage_pct
origin_bucket,,,,,,,,
1-15d,1474,"1,270,980.67",20.49,29.78,"278,270.22",21.89,0.00,100.00
16-30d,1520,"1,215,472.82",13.36,18.09,"159,862.09",13.15,10.33,100.00
31-45d,1318,"1,026,239.53",7.51,10.62,"76,241.72",7.43,92.49,100.00
46-59d,1202,"957,404.43",3.33,3.66,"22,894.38",2.39,96.67,100.00
60d+,147,"111,428.15",0.00,0.00,0.00,0.00,100.00,100.00


In [33]:
# ============================================================
# 15. SEPTEMBER MONETARY RECOVERY SCENARIO
# ============================================================

# Historical BALANCE recovery rates observed Jul → Aug
historical_balance_recovery = {
    "1-15d":  0.2189,
    "16-30d": 0.1315,
    "31-45d": 0.0743,
    "46-59d": 0.0239,
    "60d+":   0.0000
}

sep_recovery = sep_portfolio.copy()

sep_recovery["historical_balance_recovery_rate"] = (
    sep_recovery.index
    .map(historical_balance_recovery)
)


# ------------------------------------------------------------
# EXPECTED / SCENARIO RECOVERY
# ------------------------------------------------------------

sep_recovery["projected_recovery_brl"] = (
    sep_recovery["balance_brl"]
    * sep_recovery["historical_balance_recovery_rate"]
)

sep_recovery["projected_remaining_balance_brl"] = (
    sep_recovery["balance_brl"]
    - sep_recovery["projected_recovery_brl"]
)


# ------------------------------------------------------------
# SHARE OF PROJECTED RECOVERY
# ------------------------------------------------------------

sep_recovery["share_projected_recovery_pct"] = (
    sep_recovery["projected_recovery_brl"]
    / sep_recovery["projected_recovery_brl"].sum()
    * 100
)


# ------------------------------------------------------------
# PRESENTATION VIEW
# ------------------------------------------------------------

sep_recovery_view = sep_recovery[
    [
        "customers",
        "balance_brl",
        "historical_balance_recovery_rate",
        "projected_recovery_brl",
        "projected_remaining_balance_brl",
        "share_projected_recovery_pct"
    ]
].copy()

sep_recovery_view[
    "historical_balance_recovery_rate"
] *= 100


print("=" * 110)
print("SEPTEMBER EXISTING PORTFOLIO — MONETARY RECOVERY SCENARIO")
print("=" * 110)

display(
    sep_recovery_view.round(2)
)

SEPTEMBER EXISTING PORTFOLIO — MONETARY RECOVERY SCENARIO


,customers,balance_brl,historical_balance_recovery_rate,projected_recovery_brl,projected_remaining_balance_brl,share_projected_recovery_pct
sep_bucket,,,,,,
1-15d,1559,"1,280,571.56",21.89,"280,317.11","1,000,254.45",51.78
16-30d,1432,"1,192,227.01",13.15,"156,777.85","1,035,449.16",28.96
31-45d,1340,"1,090,654.81",7.43,"81,035.65","1,009,619.16",14.97
46-59d,1243,"971,487.73",2.39,"23,218.56","948,269.17",4.29
60d+,84,"66,708.45",0.00,0.00,"66,708.45",0.00


In [34]:
# ============================================================
# 15. SEPTEMBER MONETARY RECOVERY SCENARIO
# ============================================================

# Historical BALANCE recovery rates observed Jul → Aug
historical_balance_recovery = {
    "1-15d":  0.2189,
    "16-30d": 0.1315,
    "31-45d": 0.0743,
    "46-59d": 0.0239,
    "60d+":   0.0000
}

sep_recovery = sep_portfolio.copy()

sep_recovery["historical_balance_recovery_rate"] = (
    sep_recovery.index
    .map(historical_balance_recovery)
)


# ------------------------------------------------------------
# EXPECTED / SCENARIO RECOVERY
# ------------------------------------------------------------

sep_recovery["projected_recovery_brl"] = (
    sep_recovery["balance_brl"]
    * sep_recovery["historical_balance_recovery_rate"]
)

sep_recovery["projected_remaining_balance_brl"] = (
    sep_recovery["balance_brl"]
    - sep_recovery["projected_recovery_brl"]
)


# ------------------------------------------------------------
# SHARE OF PROJECTED RECOVERY
# ------------------------------------------------------------

sep_recovery["share_projected_recovery_pct"] = (
    sep_recovery["projected_recovery_brl"]
    / sep_recovery["projected_recovery_brl"].sum()
    * 100
)


# ------------------------------------------------------------
# PRESENTATION VIEW
# ------------------------------------------------------------

sep_recovery_view = sep_recovery[
    [
        "customers",
        "balance_brl",
        "historical_balance_recovery_rate",
        "projected_recovery_brl",
        "projected_remaining_balance_brl",
        "share_projected_recovery_pct"
    ]
].copy()

sep_recovery_view[
    "historical_balance_recovery_rate"
] *= 100


print("=" * 110)
print("SEPTEMBER EXISTING PORTFOLIO — MONETARY RECOVERY SCENARIO")
print("=" * 110)

display(
    sep_recovery_view.round(2)
)

SEPTEMBER EXISTING PORTFOLIO — MONETARY RECOVERY SCENARIO


,customers,balance_brl,historical_balance_recovery_rate,projected_recovery_brl,projected_remaining_balance_brl,share_projected_recovery_pct
sep_bucket,,,,,,
1-15d,1559,"1,280,571.56",21.89,"280,317.11","1,000,254.45",51.78
16-30d,1432,"1,192,227.01",13.15,"156,777.85","1,035,449.16",28.96
31-45d,1340,"1,090,654.81",7.43,"81,035.65","1,009,619.16",14.97
46-59d,1243,"971,487.73",2.39,"23,218.56","948,269.17",4.29
60d+,84,"66,708.45",0.00,0.00,"66,708.45",0.00


In [35]:
# ============================================================
# 16. EXECUTIVE RECOVERY SUMMARY
# ============================================================

total_balance = (
    sep_recovery["balance_brl"]
    .sum()
)

projected_recovery = (
    sep_recovery["projected_recovery_brl"]
    .sum()
)

remaining_balance = (
    sep_recovery["projected_remaining_balance_brl"]
    .sum()
)

portfolio_recovery_rate = (
    projected_recovery
    / total_balance
    * 100
)


print("=" * 85)
print("SEPTEMBER — EXISTING PORTFOLIO RECOVERY SCENARIO")
print("=" * 85)

print(
    f"Customers                    : "
    f"{sep_recovery['customers'].sum():,.0f}"
)

print(
    f"Starting balance             : "
    f"R$ {total_balance:,.2f}"
)

print()

print(
    f"Projected observed recovery  : "
    f"R$ {projected_recovery:,.2f}"
)

print(
    f"Portfolio recovery rate      : "
    f"{portfolio_recovery_rate:.2f}%"
)

print(
    f"Remaining balance            : "
    f"R$ {remaining_balance:,.2f}"
)

SEPTEMBER — EXISTING PORTFOLIO RECOVERY SCENARIO
Customers                    : 5,658
Starting balance             : R$ 4,601,649.56

Projected observed recovery  : R$ 541,349.18
Portfolio recovery rate      : 11.76%
Remaining balance            : R$ 4,060,300.38


In [36]:
# ============================================================
# NEW SEPTEMBER VINTAGE — ENTRY PROFILE
# ============================================================

queue["in_collections_since"] = pd.to_datetime(
    queue["in_collections_since"]
)

sep_start = pd.Timestamp("2026-09-01")
sep_end   = pd.Timestamp("2026-09-30")

new_sep = queue[
    queue["in_collections_since"] >= sep_start
].copy()

new_sep["days_available_in_sep"] = (
    sep_end - new_sep["in_collections_since"].dt.normalize()
).dt.days + 1

print("=" * 80)
print("NEW SEPTEMBER VINTAGE")
print("=" * 80)

print(f"Customers : {new_sep['customer_id'].nunique():,}")
print(
    f"Balance   : "
    f"R$ {new_sep['outstanding_balance_brl'].sum():,.2f}"
)

print("\nEntry dates:")
print(
    new_sep["in_collections_since"]
    .dt.normalize()
    .describe()
)

print("\nDays available in September:")
print(
    new_sep["days_available_in_sep"]
    .describe(
        percentiles=[.10,.25,.50,.75,.90]
    )
)

NEW SEPTEMBER VINTAGE
Customers : 5,000
Balance   : R$ 4,284,358.42

Entry dates:
count                          5000
mean     2026-09-15 12:20:09.600000
min             2026-09-01 00:00:00
25%             2026-09-08 00:00:00
50%             2026-09-15 00:00:00
75%             2026-09-23 00:00:00
max             2026-09-30 00:00:00
Name: in_collections_since, dtype: object

Days available in September:
count   5,000.00
mean       15.49
std         8.58
min         1.00
10%         4.00
25%         8.00
50%        16.00
75%        23.00
90%        27.00
max        30.00
Name: days_available_in_sep, dtype: float64


In [37]:
# ============================================================
# NEW VINTAGE — MATURITY WINDOWS
# ============================================================

new_sep["maturity_window"] = pd.cut(
    new_sep["days_available_in_sep"],
    bins=[0, 7, 15, 30, 999],
    labels=[
        "≤7 days",
        "8-15 days",
        "16-30 days",
        "30+ days"
    ]
)

new_vintage_profile = (
    new_sep
    .groupby(
        "maturity_window",
        observed=False
    )
    .agg(
        customers=("customer_id", "nunique"),
        balance_brl=("outstanding_balance_brl", "sum"),
        avg_balance_brl=("outstanding_balance_brl", "mean"),
        avg_days_available=("days_available_in_sep", "mean")
    )
)

new_vintage_profile["balance_share_pct"] = (
    new_vintage_profile["balance_brl"]
    / new_vintage_profile["balance_brl"].sum()
    * 100
)

display(
    new_vintage_profile.round(2)
)

,customers,balance_brl,avg_balance_brl,avg_days_available,balance_share_pct
maturity_window,,,,,
≤7 days,1172,"996,812.56",850.52,4.11,23.27
8-15 days,1319,"1,128,640.19",855.68,11.53,26.34
16-30 days,2509,"2,158,905.67",860.46,22.88,50.39
30+ days,0,0.00,NaN,NaN,0.00


In [38]:
# ============================================================
# HISTORICAL EARLY-COLLECTIONS RECOVERY CURVE
# D+7 / D+15 / D+30
# ============================================================

import pandas as pd
import numpy as np

df = wa.copy()

df["sent_at"] = pd.to_datetime(df["sent_at"])
df["amount_paid_brl"] = df["amount_paid_brl"].fillna(0)
df["paid_within_72h"] = df["paid_within_72h"].fillna(0).astype(int)

# ------------------------------------------------------------
# 1. RECONSTRUCT DPD AT EACH MESSAGE
# ------------------------------------------------------------

df["sent_date"] = df["sent_at"].dt.normalize()

df["implied_due_date"] = (
    df["sent_date"]
    - pd.to_timedelta(df["days_past_due"], unit="D")
)

# One due date per customer
due_customer = (
    df.groupby("customer_id", as_index=False)
      .agg(
          implied_due_date=("implied_due_date", "median"),
          first_seen=("sent_at", "min")
      )
)

# ------------------------------------------------------------
# 2. DEFINE HISTORICAL EARLY-COLLECTIONS ENTRY
#
# We use the first observation at DPD <= 7.
# This avoids pretending first WhatsApp = collections entry.
# ------------------------------------------------------------

early = df[
    (df["days_past_due"] >= 1) &
    (df["days_past_due"] <= 7)
].copy()

entry = (
    early.sort_values("sent_at")
         .groupby("customer_id", as_index=False)
         .first()[
             [
                 "customer_id",
                 "sent_at",
                 "days_past_due",
                 "outstanding_balance_brl"
             ]
         ]
         .rename(
             columns={
                 "sent_at": "entry_at",
                 "days_past_due": "entry_dpd",
                 "outstanding_balance_brl": "entry_balance_brl"
             }
         )
)

print("=" * 90)
print("HISTORICAL EARLY-COLLECTIONS COHORT")
print("=" * 90)

print(f"Customers entering analysis : {entry['customer_id'].nunique():,}")
print(
    f"Starting balance            : "
    f"R$ {entry['entry_balance_brl'].sum():,.2f}"
)

print("\nEntry DPD:")
print(entry["entry_dpd"].describe())


# ------------------------------------------------------------
# 3. PAYMENT EVENTS
# ------------------------------------------------------------

payments = df[
    (df["paid_within_72h"] == 1) &
    (df["amount_paid_brl"] > 0)
][
    [
        "customer_id",
        "sent_at",
        "amount_paid_brl"
    ]
].copy()

payments = payments.merge(
    entry[
        [
            "customer_id",
            "entry_at",
            "entry_balance_brl"
        ]
    ],
    on="customer_id",
    how="inner"
)

# Only payments after cohort entry
payments = payments[
    payments["sent_at"] >= payments["entry_at"]
].copy()

payments["days_from_entry"] = (
    payments["sent_at"].dt.normalize()
    - payments["entry_at"].dt.normalize()
).dt.days


# ------------------------------------------------------------
# 4. FUNCTION TO CALCULATE CUMULATIVE RECOVERY
# ------------------------------------------------------------

def recovery_at_horizon(entry_df, payments_df, horizon):

    # Customers with enough observable history
    observation_end = df["sent_at"].max().normalize()

    eligible = entry_df[
        entry_df["entry_at"].dt.normalize()
        <= observation_end - pd.Timedelta(days=horizon)
    ].copy()

    eligible_ids = eligible["customer_id"]

    p = payments_df[
        payments_df["customer_id"].isin(eligible_ids)
        & (payments_df["days_from_entry"] >= 0)
        & (payments_df["days_from_entry"] <= horizon)
    ].copy()

    recovered_customer = (
        p.groupby("customer_id", as_index=False)
         .agg(
             recovered_brl=("amount_paid_brl", "sum")
         )
    )

    result = eligible[
        [
            "customer_id",
            "entry_balance_brl"
        ]
    ].merge(
        recovered_customer,
        on="customer_id",
        how="left"
    )

    result["recovered_brl"] = (
        result["recovered_brl"]
        .fillna(0)
    )

    # Cap recovery at starting balance
    result["recovered_brl"] = np.minimum(
        result["recovered_brl"],
        result["entry_balance_brl"]
    )

    result["had_payment"] = (
        result["recovered_brl"] > 0
    ).astype(int)

    customers = len(result)

    starting_balance = (
        result["entry_balance_brl"].sum()
    )

    recovered = (
        result["recovered_brl"].sum()
    )

    payers = (
        result["had_payment"].sum()
    )

    recovery_rate = (
        recovered / starting_balance
        if starting_balance > 0
        else np.nan
    )

    payer_rate = (
        payers / customers
        if customers > 0
        else np.nan
    )

    return {
        "horizon": f"D+{horizon}",
        "customers": customers,
        "starting_balance_brl": starting_balance,
        "payers": payers,
        "payer_rate_pct": payer_rate * 100,
        "recovered_brl": recovered,
        "balance_recovery_rate_pct": recovery_rate * 100
    }


# ------------------------------------------------------------
# 5. D+7 / D+15 / D+30
# ------------------------------------------------------------

curve = pd.DataFrame(
    [
        recovery_at_horizon(entry, payments, 7),
        recovery_at_horizon(entry, payments, 15),
        recovery_at_horizon(entry, payments, 30)
    ]
)

print("\n")
print("=" * 90)
print("HISTORICAL CUMULATIVE RECOVERY CURVE")
print("=" * 90)

display(
    curve.style.format(
        {
            "customers": "{:,.0f}",
            "starting_balance_brl": "R$ {:,.2f}",
            "payers": "{:,.0f}",
            "payer_rate_pct": "{:.2f}%",
            "recovered_brl": "R$ {:,.2f}",
            "balance_recovery_rate_pct": "{:.2f}%"
        }
    )
)


# ------------------------------------------------------------
# 6. INCREMENTAL RECOVERY BETWEEN WINDOWS
# ------------------------------------------------------------

curve_incremental = curve[
    [
        "horizon",
        "balance_recovery_rate_pct"
    ]
].copy()

curve_incremental["incremental_recovery_pp"] = (
    curve_incremental[
        "balance_recovery_rate_pct"
    ].diff()
)

curve_incremental.loc[
    curve_incremental.index[0],
    "incremental_recovery_pp"
] = curve_incremental.loc[
    curve_incremental.index[0],
    "balance_recovery_rate_pct"
]

print("\n")
print("=" * 90)
print("INCREMENTAL RECOVERY BY MATURITY WINDOW")
print("=" * 90)

display(
    curve_incremental.style.format(
        {
            "balance_recovery_rate_pct": "{:.2f}%",
            "incremental_recovery_pp": "{:.2f} pp"
        }
    )
)

HISTORICAL EARLY-COLLECTIONS COHORT
Customers entering analysis : 11,019
Starting balance            : R$ 9,367,043.26

Entry DPD:
count   11,019.00
mean         2.61
std          1.68
min          1.00
25%          1.00
50%          2.00
75%          4.00
max          7.00
Name: entry_dpd, dtype: float64


HISTORICAL CUMULATIVE RECOVERY CURVE


,horizon,customers,starting_balance_brl,payers,payer_rate_pct,recovered_brl,balance_recovery_rate_pct
0,D+7,"10,163","R$ 8,638,324.75","2,511",24.71%,"R$ 1,687,257.69",19.53%
1,D+15,"9,142","R$ 7,766,044.16","3,102",33.93%,"R$ 2,139,402.45",27.55%
2,D+30,"7,454","R$ 6,297,641.60","3,109",41.71%,"R$ 2,196,711.02",34.88%




INCREMENTAL RECOVERY BY MATURITY WINDOW


,horizon,balance_recovery_rate_pct,incremental_recovery_pp
0,D+7,19.53%,19.53 pp
1,D+15,27.55%,8.02 pp
2,D+30,34.88%,7.33 pp


In [39]:
# ============================================================
# SEPTEMBER NEW VINTAGE
# MATURITY-ADJUSTED RECOVERY SCENARIO
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. HISTORICAL MATURITY CURVE
# ------------------------------------------------------------

curve_days = np.array([
    0,
    7,
    15,
    30
])

curve_rates = np.array([
    0.00,
    0.1953,
    0.2755,
    0.3488
])


# ------------------------------------------------------------
# 2. APPLY CURVE TO EACH NEW CUSTOMER
# ------------------------------------------------------------

new_sep_projection = new_sep.copy()

new_sep_projection[
    "historical_recovery_rate"
] = np.interp(
    new_sep_projection["days_available_in_sep"],
    curve_days,
    curve_rates
)


# ------------------------------------------------------------
# 3. PROJECT MONETARY RECOVERY
# ------------------------------------------------------------

new_sep_projection[
    "projected_recovery_brl"
] = (
    new_sep_projection["outstanding_balance_brl"]
    *
    new_sep_projection["historical_recovery_rate"]
)

new_sep_projection[
    "projected_remaining_balance_brl"
] = (
    new_sep_projection["outstanding_balance_brl"]
    -
    new_sep_projection["projected_recovery_brl"]
)


# ------------------------------------------------------------
# 4. PORTFOLIO SUMMARY
# ------------------------------------------------------------

new_customers = (
    new_sep_projection["customer_id"]
    .nunique()
)

new_balance = (
    new_sep_projection["outstanding_balance_brl"]
    .sum()
)

new_projected_recovery = (
    new_sep_projection["projected_recovery_brl"]
    .sum()
)

new_remaining_balance = (
    new_sep_projection[
        "projected_remaining_balance_brl"
    ].sum()
)

new_recovery_rate = (
    new_projected_recovery
    / new_balance
    * 100
)


print("=" * 90)
print("SEPTEMBER NEW VINTAGE — MATURITY-ADJUSTED RECOVERY")
print("=" * 90)

print(
    f"Customers                  : "
    f"{new_customers:,}"
)

print(
    f"Starting balance           : "
    f"R$ {new_balance:,.2f}"
)

print(
    f"Projected recovery         : "
    f"R$ {new_projected_recovery:,.2f}"
)

print(
    f"Projected recovery rate    : "
    f"{new_recovery_rate:.2f}%"
)

print(
    f"Projected remaining balance: "
    f"R$ {new_remaining_balance:,.2f}"
)


# ------------------------------------------------------------
# 5. SUMMARY BY MATURITY WINDOW
# ------------------------------------------------------------

new_sep_projection["maturity_window"] = pd.cut(
    new_sep_projection["days_available_in_sep"],
    bins=[0, 7, 15, 30],
    labels=[
        "≤7 days",
        "8-15 days",
        "16-30 days"
    ],
    include_lowest=True
)

maturity_projection = (
    new_sep_projection
    .groupby(
        "maturity_window",
        observed=False
    )
    .agg(
        customers=(
            "customer_id",
            "nunique"
        ),
        balance_brl=(
            "outstanding_balance_brl",
            "sum"
        ),
        avg_days_available=(
            "days_available_in_sep",
            "mean"
        ),
        avg_recovery_rate=(
            "historical_recovery_rate",
            "mean"
        ),
        projected_recovery_brl=(
            "projected_recovery_brl",
            "sum"
        )
    )
)

maturity_projection[
    "effective_recovery_rate"
] = (
    maturity_projection[
        "projected_recovery_brl"
    ]
    /
    maturity_projection[
        "balance_brl"
    ]
)

maturity_projection[
    "share_recovery_pct"
] = (
    maturity_projection[
        "projected_recovery_brl"
    ]
    /
    maturity_projection[
        "projected_recovery_brl"
    ].sum()
    * 100
)


display(
    maturity_projection.style.format(
        {
            "customers": "{:,.0f}",
            "balance_brl": "R$ {:,.2f}",
            "avg_days_available": "{:.1f}",
            "avg_recovery_rate": "{:.2%}",
            "projected_recovery_brl": "R$ {:,.2f}",
            "effective_recovery_rate": "{:.2%}",
            "share_recovery_pct": "{:.2f}%"
        }
    )
)

SEPTEMBER NEW VINTAGE — MATURITY-ADJUSTED RECOVERY
Customers                  : 5,000
Starting balance           : R$ 4,284,358.42
Projected recovery         : R$ 1,066,701.36
Projected recovery rate    : 24.90%
Projected remaining balance: R$ 3,217,657.06


,customers,balance_brl,avg_days_available,avg_recovery_rate,projected_recovery_brl,effective_recovery_rate,share_recovery_pct
maturity_window,,,,,,,
≤7 days,"1,172","R$ 996,812.56",4.1,11.46%,"R$ 116,453.62",11.68%,10.92%
8-15 days,"1,319","R$ 1,128,640.19",11.5,24.07%,"R$ 272,648.94",24.16%,25.56%
16-30 days,"2,509","R$ 2,158,905.67",22.9,31.40%,"R$ 677,598.81",31.39%,63.52%


Em qual estágio de DPD o dinheiro é efetivamente recuperado?

In [40]:
# ============================================================
# DPD-BASED MONETARY RECOVERY
# Recovery incremental por estágio de inadimplência
# ============================================================

import pandas as pd
import numpy as np

df = wa.copy()

df["sent_at"] = pd.to_datetime(df["sent_at"])
df["amount_paid_brl"] = df["amount_paid_brl"].fillna(0)
df["paid_within_72h"] = df["paid_within_72h"].fillna(0).astype(int)

# ------------------------------------------------------------
# 1. DPD BUCKET
# ------------------------------------------------------------

def dpd_stage(dpd):

    if pd.isna(dpd):
        return np.nan

    if 1 <= dpd <= 7:
        return "01-07"

    elif 8 <= dpd <= 15:
        return "08-15"

    elif 16 <= dpd <= 30:
        return "16-30"

    elif 31 <= dpd <= 45:
        return "31-45"

    elif 46 <= dpd <= 59:
        return "46-59"

    elif dpd >= 60:
        return "60+"

    return np.nan


df["dpd_stage"] = df["days_past_due"].apply(dpd_stage)

stage_order = [
    "01-07",
    "08-15",
    "16-30",
    "31-45",
    "46-59",
    "60+"
]


# ------------------------------------------------------------
# 2. PAYMENT EVENTS
#
# O pagamento pertence ao DPD da mensagem que gerou
# paid_within_72h.
# ------------------------------------------------------------

payments = df[
    (df["paid_within_72h"] == 1) &
    (df["amount_paid_brl"] > 0) &
    (df["dpd_stage"].notna())
].copy()


# ------------------------------------------------------------
# 3. RECOVERY R$ BY DPD
#
# Cada pagamento aparece UMA única vez.
# ------------------------------------------------------------

recovery_by_dpd = (
    payments
    .groupby(
        "dpd_stage",
        observed=False
    )
    .agg(
        payment_events=(
            "amount_paid_brl",
            "size"
        ),

        paying_customers=(
            "customer_id",
            "nunique"
        ),

        recovered_brl=(
            "amount_paid_brl",
            "sum"
        )
    )
    .reindex(stage_order)
    .fillna(0)
)


recovery_by_dpd[
    "share_total_recovery_pct"
] = (
    recovery_by_dpd["recovered_brl"]
    /
    recovery_by_dpd["recovered_brl"].sum()
    * 100
)


# ------------------------------------------------------------
# 4. CUSTOMERS / BALANCE EXPOSED TO EACH DPD STAGE
#
# Para cada cliente e bucket:
# usamos a PRIMEIRA observação naquele estágio.
#
# Assim o denominador representa o saldo quando ele
# entrou naquele estágio de DPD.
# ------------------------------------------------------------

stage_entry = (
    df[
        df["dpd_stage"].notna()
    ]
    .sort_values(
        [
            "customer_id",
            "sent_at"
        ]
    )
    .groupby(
        [
            "customer_id",
            "dpd_stage"
        ],
        observed=False,
        as_index=False
    )
    .first()
)


exposure_by_dpd = (
    stage_entry
    .groupby(
        "dpd_stage",
        observed=False
    )
    .agg(
        customers_exposed=(
            "customer_id",
            "nunique"
        ),

        starting_balance_brl=(
            "outstanding_balance_brl",
            "sum"
        )
    )
    .reindex(stage_order)
)


# ------------------------------------------------------------
# 5. COMBINE EXPOSURE + RECOVERY
# ------------------------------------------------------------

dpd_recovery = (
    exposure_by_dpd
    .join(
        recovery_by_dpd,
        how="left"
    )
    .fillna(0)
)


dpd_recovery[
    "payer_rate_pct"
] = (
    dpd_recovery["paying_customers"]
    /
    dpd_recovery["customers_exposed"]
    * 100
)


dpd_recovery[
    "incremental_recovery_rate_pct"
] = (
    dpd_recovery["recovered_brl"]
    /
    dpd_recovery["starting_balance_brl"]
    * 100
)


# ------------------------------------------------------------
# 6. DISPLAY
# ------------------------------------------------------------

print("=" * 110)

print(
    "INCREMENTAL MONETARY RECOVERY BY DPD STAGE"
)

print("=" * 110)


display(
    dpd_recovery.style.format(
        {
            "customers_exposed": "{:,.0f}",

            "starting_balance_brl":
                "R$ {:,.2f}",

            "payment_events":
                "{:,.0f}",

            "paying_customers":
                "{:,.0f}",

            "payer_rate_pct":
                "{:.2f}%",

            "recovered_brl":
                "R$ {:,.2f}",

            "incremental_recovery_rate_pct":
                "{:.2f}%",

            "share_total_recovery_pct":
                "{:.2f}%"
        }
    )
)


# ------------------------------------------------------------
# 7. CHECK — NO DOUBLE COUNTING OF PAYMENT EVENTS
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("RECONCILIATION")
print("=" * 110)

print(
    f"Payment events assigned to DPD buckets : "
    f"{int(dpd_recovery['payment_events'].sum()):,}"
)

print(
    f"Recovered assigned to DPD buckets      : "
    f"R$ {dpd_recovery['recovered_brl'].sum():,.2f}"
)

print(
    f"Total payment events in source         : "
    f"{len(payments):,}"
)

print(
    f"Total recovered in source              : "
    f"R$ {payments['amount_paid_brl'].sum():,.2f}"
)

INCREMENTAL MONETARY RECOVERY BY DPD STAGE


,customers_exposed,starting_balance_brl,payment_events,paying_customers,recovered_brl,share_total_recovery_pct,payer_rate_pct,incremental_recovery_rate_pct
dpd_stage,,,,,,,,
01-07,"11,019","R$ 9,367,043.26","2,150","2,122","R$ 1,401,097.17",40.50%,19.26%,14.96%
08-15,"9,046","R$ 7,471,051.54","1,639","1,610","R$ 1,009,350.30",29.18%,17.80%,13.51%
16-30,"7,179","R$ 5,784,111.66","1,060","1,042","R$ 617,728.45",17.86%,14.51%,10.68%
31-45,"4,269","R$ 3,375,527.61",548,539,"R$ 305,876.07",8.84%,12.63%,9.06%
46-59,"2,804","R$ 2,185,412.75",219,219,"R$ 118,018.62",3.41%,7.81%,5.40%
60+,242,"R$ 190,227.62",10,10,"R$ 7,234.69",0.21%,4.13%,3.80%



RECONCILIATION
Payment events assigned to DPD buckets : 5,626
Recovered assigned to DPD buckets      : R$ 3,459,305.30
Total payment events in source         : 5,626
Total recovered in source              : R$ 3,459,305.30


In [41]:
# ============================================================
# MONTHLY RECOVERY DISTRIBUTION BY DPD
# Jun / Jul / Aug
#
# Pergunta:
# Em cada mês, onde no aging (DPD) aconteceu o recovery?
#
# IMPORTANTE:
# - cada evento de pagamento aparece UMA única vez
# - pagamento é atribuído ao DPD da mensagem associada
# - partial payments podem aparecer em buckets diferentes
# - NÃO estamos calculando ainda "probabilidade de recovery"
# ============================================================

import pandas as pd
import numpy as np

df = wa.copy()

df["sent_at"] = pd.to_datetime(df["sent_at"])
df["amount_paid_brl"] = df["amount_paid_brl"].fillna(0)
df["paid_within_72h"] = (
    df["paid_within_72h"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 1. MONTH
# ------------------------------------------------------------

df["month"] = df["sent_at"].dt.to_period("M")

month_labels = {
    pd.Period("2026-06"): "Jun",
    pd.Period("2026-07"): "Jul",
    pd.Period("2026-08"): "Aug",
}

df["month_label"] = df["month"].map(month_labels)


# ------------------------------------------------------------
# 2. DPD STAGE
# ------------------------------------------------------------

def dpd_stage(dpd):

    if pd.isna(dpd):
        return np.nan

    if 1 <= dpd <= 7:
        return "01-07"

    elif 8 <= dpd <= 15:
        return "08-15"

    elif 16 <= dpd <= 30:
        return "16-30"

    elif 31 <= dpd <= 45:
        return "31-45"

    elif 46 <= dpd <= 59:
        return "46-59"

    elif dpd >= 60:
        return "60+"

    return np.nan


df["dpd_stage"] = df["days_past_due"].apply(dpd_stage)

stage_order = [
    "01-07",
    "08-15",
    "16-30",
    "31-45",
    "46-59",
    "60+"
]


# ------------------------------------------------------------
# 3. PAYMENT EVENTS
# ------------------------------------------------------------

payments = df[
    (df["paid_within_72h"] == 1)
    & (df["amount_paid_brl"] > 0)
    & (df["dpd_stage"].notna())
    & (df["month_label"].notna())
].copy()


# ------------------------------------------------------------
# 4. MONTH × DPD
# ------------------------------------------------------------

monthly_dpd = (
    payments
    .groupby(
        ["month_label", "dpd_stage"],
        observed=False
    )
    .agg(
        payment_events=(
            "amount_paid_brl",
            "size"
        ),

        paying_customers=(
            "customer_id",
            "nunique"
        ),

        recovered_brl=(
            "amount_paid_brl",
            "sum"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. TOTAL RECOVERY BY MONTH
# ------------------------------------------------------------

monthly_total = (
    monthly_dpd
    .groupby("month_label")["recovered_brl"]
    .sum()
    .rename("monthly_recovery_brl")
    .reset_index()
)


monthly_dpd = monthly_dpd.merge(
    monthly_total,
    on="month_label",
    how="left"
)


# ------------------------------------------------------------
# 6. SHARE OF MONTHLY RECOVERY
# ------------------------------------------------------------

monthly_dpd["share_monthly_recovery_pct"] = (
    monthly_dpd["recovered_brl"]
    /
    monthly_dpd["monthly_recovery_brl"]
    * 100
)


# ------------------------------------------------------------
# 7. ORDER
# ------------------------------------------------------------

month_order = ["Jun", "Jul", "Aug"]

monthly_dpd["month_label"] = pd.Categorical(
    monthly_dpd["month_label"],
    categories=month_order,
    ordered=True
)

monthly_dpd["dpd_stage"] = pd.Categorical(
    monthly_dpd["dpd_stage"],
    categories=stage_order,
    ordered=True
)

monthly_dpd = (
    monthly_dpd
    .sort_values(
        ["month_label", "dpd_stage"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. DISPLAY — FULL TABLE
# ------------------------------------------------------------

print("=" * 105)
print("MONTHLY MONETARY RECOVERY BY DPD")
print("=" * 105)

display(
    monthly_dpd[
        [
            "month_label",
            "dpd_stage",
            "payment_events",
            "paying_customers",
            "recovered_brl",
            "share_monthly_recovery_pct"
        ]
    ].style.format(
        {
            "payment_events": "{:,.0f}",
            "paying_customers": "{:,.0f}",
            "recovered_brl": "R$ {:,.2f}",
            "share_monthly_recovery_pct": "{:.2f}%"
        }
    )
)


# ============================================================
# 9. PIVOT — % OF RECOVERY BY DPD
# ============================================================

share_pivot = (
    monthly_dpd
    .pivot(
        index="dpd_stage",
        columns="month_label",
        values="share_monthly_recovery_pct"
    )
    .reindex(stage_order)
)

print("\n" + "=" * 105)
print("SHARE OF MONTHLY RECOVERY BY DPD")
print("=" * 105)

display(
    share_pivot.style.format("{:.2f}%")
)


# ============================================================
# 10. PIVOT — R$ RECOVERED
# ============================================================

recovery_pivot = (
    monthly_dpd
    .pivot(
        index="dpd_stage",
        columns="month_label",
        values="recovered_brl"
    )
    .reindex(stage_order)
)

print("\n" + "=" * 105)
print("RECOVERED R$ BY MONTH × DPD")
print("=" * 105)

display(
    recovery_pivot.style.format("R$ {:,.2f}")
)


# ============================================================
# 11. KEY KPI — RECOVERY AT DPD <= 30
# ============================================================

early = monthly_dpd[
    monthly_dpd["dpd_stage"].isin(
        ["01-07", "08-15", "16-30"]
    )
].copy()

early_monthly = (
    early
    .groupby(
        "month_label",
        observed=False
    )
    .agg(
        recovery_dpd_le_30_brl=(
            "recovered_brl",
            "sum"
        )
    )
    .reset_index()
)

early_monthly = early_monthly.merge(
    monthly_total,
    on="month_label",
    how="left"
)

early_monthly["share_recovery_dpd_le_30_pct"] = (
    early_monthly["recovery_dpd_le_30_brl"]
    /
    early_monthly["monthly_recovery_brl"]
    * 100
)

early_monthly["recovery_after_30_brl"] = (
    early_monthly["monthly_recovery_brl"]
    -
    early_monthly["recovery_dpd_le_30_brl"]
)

early_monthly["share_recovery_after_30_pct"] = (
    100
    -
    early_monthly["share_recovery_dpd_le_30_pct"]
)


print("\n" + "=" * 105)
print("EARLY RECOVERY CONCENTRATION — DPD <= 30")
print("=" * 105)

display(
    early_monthly.style.format(
        {
            "recovery_dpd_le_30_brl":
                "R$ {:,.2f}",

            "monthly_recovery_brl":
                "R$ {:,.2f}",

            "share_recovery_dpd_le_30_pct":
                "{:.2f}%",

            "recovery_after_30_brl":
                "R$ {:,.2f}",

            "share_recovery_after_30_pct":
                "{:.2f}%"
        }
    )
)


# ============================================================
# 12. STABILITY SUMMARY
# ============================================================

stability = (
    early_monthly[
        "share_recovery_dpd_le_30_pct"
    ]
    .agg(
        ["mean", "min", "max", "std"]
    )
)

print("\n" + "=" * 105)
print("STABILITY OF DPD <= 30 RECOVERY SHARE")
print("=" * 105)

print(
    f"Average : {stability['mean']:.2f}%"
)

print(
    f"Minimum : {stability['min']:.2f}%"
)

print(
    f"Maximum : {stability['max']:.2f}%"
)

print(
    f"Std dev : {stability['std']:.2f} pp"
)


# ============================================================
# 13. RECONCILIATION
# ============================================================

print("\n" + "=" * 105)
print("RECONCILIATION")
print("=" * 105)

print(
    f"Payment events : "
    f"{monthly_dpd['payment_events'].sum():,.0f}"
)

print(
    f"Recovered      : "
    f"R$ {monthly_dpd['recovered_brl'].sum():,.2f}"
)

print(
    f"Source events  : "
    f"{len(payments):,.0f}"
)

print(
    f"Source recovery: "
    f"R$ {payments['amount_paid_brl'].sum():,.2f}"
)

MONTHLY MONETARY RECOVERY BY DPD


,month_label,dpd_stage,payment_events,paying_customers,recovered_brl,share_monthly_recovery_pct
0,Jun,01-07,689,679,"R$ 461,427.25",59.61%
1,Jun,08-15,385,378,"R$ 227,123.08",29.34%
2,Jun,16-30,128,127,"R$ 85,492.71",11.04%
3,Jul,01-07,740,732,"R$ 478,549.95",35.69%
4,Jul,08-15,648,636,"R$ 403,149.50",30.07%
5,Jul,16-30,480,471,"R$ 285,519.75",21.30%
6,Jul,31-45,251,247,"R$ 143,042.23",10.67%
7,Jul,46-59,56,56,"R$ 29,333.10",2.19%
8,Jul,60+,2,2,"R$ 1,077.45",0.08%
9,Aug,01-07,721,712,"R$ 461,119.97",34.29%



SHARE OF MONTHLY RECOVERY BY DPD


month_label,Jun,Jul,Aug
dpd_stage,,,
01-07,59.61%,35.69%,34.29%
08-15,29.34%,30.07%,28.19%
16-30,11.04%,21.30%,18.35%
31-45,nan%,10.67%,12.11%
46-59,nan%,2.19%,6.60%
60+,nan%,0.08%,0.46%



RECOVERED R$ BY MONTH × DPD


month_label,Jun,Jul,Aug
dpd_stage,,,
01-07,"R$ 461,427.25","R$ 478,549.95","R$ 461,119.97"
08-15,"R$ 227,123.08","R$ 403,149.50","R$ 379,077.72"
16-30,"R$ 85,492.71","R$ 285,519.75","R$ 246,715.99"
31-45,R$ nan,"R$ 143,042.23","R$ 162,833.84"
46-59,R$ nan,"R$ 29,333.10","R$ 88,685.52"
60+,R$ nan,"R$ 1,077.45","R$ 6,157.24"



EARLY RECOVERY CONCENTRATION — DPD <= 30


,month_label,recovery_dpd_le_30_brl,monthly_recovery_brl,share_recovery_dpd_le_30_pct,recovery_after_30_brl,share_recovery_after_30_pct
0,Jun,"R$ 774,043.04","R$ 774,043.04",100.00%,R$ 0.00,0.00%
1,Jul,"R$ 1,167,219.20","R$ 1,340,671.98",87.06%,"R$ 173,452.78",12.94%
2,Aug,"R$ 1,086,913.68","R$ 1,344,590.28",80.84%,"R$ 257,676.60",19.16%



STABILITY OF DPD <= 30 RECOVERY SHARE
Average : 89.30%
Minimum : 80.84%
Maximum : 100.00%
Std dev : 9.78 pp

RECONCILIATION
Payment events : 5,626
Recovered      : R$ 3,459,305.30
Source events  : 5,626
Source recovery: R$ 3,459,305.30


In [42]:
# ============================================================
# SLIDE 2 — CONTACT EFFORT BEFORE vs AFTER DPD30
# Jul / Aug / Jul-Aug consolidated
#
# Objetivo:
# comparar esforço de cobrança vs recovery observado
# em DPD <=30 e DPD >30
# ============================================================

import pandas as pd
import numpy as np

df = wa.copy()

df["sent_at"] = pd.to_datetime(df["sent_at"])
df["amount_paid_brl"] = df["amount_paid_brl"].fillna(0)
df["paid_within_72h"] = (
    df["paid_within_72h"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 1. MONTH
# ------------------------------------------------------------

df["month"] = df["sent_at"].dt.to_period("M")

month_map = {
    pd.Period("2026-07"): "Jul",
    pd.Period("2026-08"): "Aug"
}

df["month_label"] = df["month"].map(month_map)

# somente meses comparáveis
x = df[
    df["month_label"].isin(["Jul", "Aug"])
].copy()


# ------------------------------------------------------------
# 2. DPD WINDOW
# ------------------------------------------------------------

x["dpd_window"] = np.where(
    x["days_past_due"] <= 30,
    "DPD <=30",
    "DPD >30"
)

# remover DPD inválido / negativo, se houver
x = x[
    x["days_past_due"].notna()
    & (x["days_past_due"] >= 1)
].copy()


# ------------------------------------------------------------
# 3. PAYMENT VALUE
# ------------------------------------------------------------

x["recovery_brl"] = np.where(
    (x["paid_within_72h"] == 1)
    & (x["amount_paid_brl"] > 0),
    x["amount_paid_brl"],
    0
)


# ------------------------------------------------------------
# 4. MONTH × DPD WINDOW
# ------------------------------------------------------------

monthly = (
    x
    .groupby(
        ["month_label", "dpd_window"],
        observed=False
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("paid_within_72h", "sum"),
        recovered_brl=("recovery_brl", "sum")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. SHARES WITHIN MONTH
# ------------------------------------------------------------

monthly["share_messages_pct"] = (
    monthly["messages"]
    /
    monthly.groupby("month_label")["messages"].transform("sum")
    * 100
)

monthly["share_recovery_pct"] = (
    monthly["recovered_brl"]
    /
    monthly.groupby("month_label")["recovered_brl"].transform("sum")
    * 100
)


# ------------------------------------------------------------
# 6. COST
# R$1 per WhatsApp message
# ------------------------------------------------------------

MESSAGE_COST = 1.0

monthly["message_cost_brl"] = (
    monthly["messages"] * MESSAGE_COST
)


# ------------------------------------------------------------
# 7. RECOVERY PER MESSAGE
#
# Não é causal ROI.
# É apenas produtividade observada do esforço.
# ------------------------------------------------------------

monthly["recovery_per_message_brl"] = (
    monthly["recovered_brl"]
    /
    monthly["messages"]
)


# ------------------------------------------------------------
# 8. RECOVERY / MESSAGE COST
#
# R$ recovered per R$1 spent
# ------------------------------------------------------------

monthly["recovery_per_cost_brl"] = (
    monthly["recovered_brl"]
    /
    monthly["message_cost_brl"]
)


# ------------------------------------------------------------
# 9. DISPLAY MONTHLY
# ------------------------------------------------------------

print("=" * 115)
print("CONTACT EFFORT vs RECOVERY — JUL / AUG")
print("=" * 115)

display(
    monthly.style.format(
        {
            "messages": "{:,.0f}",
            "customers": "{:,.0f}",
            "payment_events": "{:,.0f}",
            "recovered_brl": "R$ {:,.2f}",
            "share_messages_pct": "{:.2f}%",
            "share_recovery_pct": "{:.2f}%",
            "message_cost_brl": "R$ {:,.2f}",
            "recovery_per_message_brl": "R$ {:,.2f}",
            "recovery_per_cost_brl": "R$ {:,.2f}"
        }
    )
)


# ============================================================
# 10. JUL–AUG CONSOLIDATED
# ============================================================

consolidated = (
    x
    .groupby(
        "dpd_window",
        observed=False
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("paid_within_72h", "sum"),
        recovered_brl=("recovery_brl", "sum")
    )
    .reset_index()
)


consolidated["share_messages_pct"] = (
    consolidated["messages"]
    /
    consolidated["messages"].sum()
    * 100
)


consolidated["share_recovery_pct"] = (
    consolidated["recovered_brl"]
    /
    consolidated["recovered_brl"].sum()
    * 100
)


consolidated["message_cost_brl"] = (
    consolidated["messages"]
    * MESSAGE_COST
)


consolidated["recovery_per_message_brl"] = (
    consolidated["recovered_brl"]
    /
    consolidated["messages"]
)


consolidated["recovery_per_cost_brl"] = (
    consolidated["recovered_brl"]
    /
    consolidated["message_cost_brl"]
)


print("\n" + "=" * 115)
print("CONTACT EFFORT vs RECOVERY — JUL–AUG CONSOLIDATED")
print("=" * 115)

display(
    consolidated.style.format(
        {
            "messages": "{:,.0f}",
            "customers": "{:,.0f}",
            "payment_events": "{:,.0f}",
            "recovered_brl": "R$ {:,.2f}",
            "share_messages_pct": "{:.2f}%",
            "share_recovery_pct": "{:.2f}%",
            "message_cost_brl": "R$ {:,.2f}",
            "recovery_per_message_brl": "R$ {:,.2f}",
            "recovery_per_cost_brl": "R$ {:,.2f}"
        }
    )
)


# ============================================================
# 11. EFFORT vs RECOVERY GAP
#
# Se share_messages > share_recovery:
# estamos usando proporcionalmente mais esforço que o
# recovery observado naquela janela.
# ============================================================

consolidated["effort_recovery_gap_pp"] = (
    consolidated["share_messages_pct"]
    -
    consolidated["share_recovery_pct"]
)


print("\n" + "=" * 115)
print("EFFORT vs RECOVERY GAP")
print("=" * 115)

display(
    consolidated[
        [
            "dpd_window",
            "share_messages_pct",
            "share_recovery_pct",
            "effort_recovery_gap_pp",
            "recovery_per_message_brl"
        ]
    ].style.format(
        {
            "share_messages_pct": "{:.2f}%",
            "share_recovery_pct": "{:.2f}%",
            "effort_recovery_gap_pp": "{:+.2f} pp",
            "recovery_per_message_brl": "R$ {:,.2f}"
        }
    )
)


# ============================================================
# 12. MONTHLY COMPACT TABLE FOR SLIDE
# ============================================================

compact = monthly[
    [
        "month_label",
        "dpd_window",
        "messages",
        "share_messages_pct",
        "recovered_brl",
        "share_recovery_pct",
        "recovery_per_message_brl"
    ]
].copy()

print("\n" + "=" * 115)
print("SLIDE TABLE")
print("=" * 115)

display(
    compact.style.format(
        {
            "messages": "{:,.0f}",
            "share_messages_pct": "{:.1f}%",
            "recovered_brl": "R$ {:,.0f}",
            "share_recovery_pct": "{:.1f}%",
            "recovery_per_message_brl": "R$ {:,.2f}"
        }
    )
)


# ============================================================
# 13. RECONCILIATION
# ============================================================

print("\n" + "=" * 115)
print("RECONCILIATION — JUL–AUG")
print("=" * 115)

print(
    f"Messages analysed : "
    f"{len(x):,.0f}"
)

print(
    f"Recovered         : "
    f"R$ {x['recovery_brl'].sum():,.2f}"
)

print(
    f"Payment events    : "
    f"{((x['paid_within_72h'] == 1) & (x['amount_paid_brl'] > 0)).sum():,.0f}"
)

CONTACT EFFORT vs RECOVERY — JUL / AUG


,month_label,dpd_window,messages,customers,payment_events,recovered_brl,share_messages_pct,share_recovery_pct,message_cost_brl,recovery_per_message_brl,recovery_per_cost_brl
0,Aug,DPD <=30,"23,222","6,295","1,779","R$ 1,086,913.68",74.37%,80.84%,"R$ 23,222.00",R$ 46.81,R$ 46.81
1,Aug,DPD >30,"8,003","3,886",468,"R$ 257,676.60",25.63%,19.16%,"R$ 8,003.00",R$ 32.20,R$ 32.20
2,Jul,DPD <=30,"25,282","6,616","1,868","R$ 1,167,219.20",85.08%,87.06%,"R$ 25,282.00",R$ 46.17,R$ 46.17
3,Jul,DPD >30,"4,434","2,131",309,"R$ 173,452.78",14.92%,12.94%,"R$ 4,434.00",R$ 39.12,R$ 39.12



CONTACT EFFORT vs RECOVERY — JUL–AUG CONSOLIDATED


,dpd_window,messages,customers,payment_events,recovered_brl,share_messages_pct,share_recovery_pct,message_cost_brl,recovery_per_message_brl,recovery_per_cost_brl
0,DPD <=30,"48,504","10,503","3,647","R$ 2,254,132.88",79.59%,83.94%,"R$ 48,504.00",R$ 46.47,R$ 46.47
1,DPD >30,"12,437","4,884",777,"R$ 431,129.38",20.41%,16.06%,"R$ 12,437.00",R$ 34.67,R$ 34.67



EFFORT vs RECOVERY GAP


,dpd_window,share_messages_pct,share_recovery_pct,effort_recovery_gap_pp,recovery_per_message_brl
0,DPD <=30,79.59%,83.94%,-4.35 pp,R$ 46.47
1,DPD >30,20.41%,16.06%,+4.35 pp,R$ 34.67



SLIDE TABLE


,month_label,dpd_window,messages,share_messages_pct,recovered_brl,share_recovery_pct,recovery_per_message_brl
0,Aug,DPD <=30,"23,222",74.4%,"R$ 1,086,914",80.8%,R$ 46.81
1,Aug,DPD >30,"8,003",25.6%,"R$ 257,677",19.2%,R$ 32.20
2,Jul,DPD <=30,"25,282",85.1%,"R$ 1,167,219",87.1%,R$ 46.17
3,Jul,DPD >30,"4,434",14.9%,"R$ 173,453",12.9%,R$ 39.12



RECONCILIATION — JUL–AUG
Messages analysed : 60,941
Recovered         : R$ 2,685,262.26
Payment events    : 4,424


In [43]:
# ============================================================
# CONTACT EFFORT BY DPD STAGE
# Jun / Jul / Aug + full period
#
# Perguntas:
# 1. Quantos clientes foram tratados em cada DPD?
# 2. Quantas mensagens foram consumidas?
# 3. Qual o share do esforço?
# 4. Quantas mensagens, em média/mediana, cada cliente recebe
#    enquanto está em cada estágio?
# 5. Quanto esforço está concentrado até DPD30?
# ============================================================

import pandas as pd
import numpy as np

df = wa.copy()

df["sent_at"] = pd.to_datetime(df["sent_at"])

# ------------------------------------------------------------
# 1. MONTH
# ------------------------------------------------------------

df["month"] = df["sent_at"].dt.to_period("M")

month_map = {
    pd.Period("2026-06"): "Jun",
    pd.Period("2026-07"): "Jul",
    pd.Period("2026-08"): "Aug"
}

df["month_label"] = df["month"].map(month_map)


# ------------------------------------------------------------
# 2. DPD STAGE
# ------------------------------------------------------------

def dpd_stage(dpd):

    if pd.isna(dpd):
        return np.nan

    if 1 <= dpd <= 7:
        return "01-07"

    elif 8 <= dpd <= 15:
        return "08-15"

    elif 16 <= dpd <= 30:
        return "16-30"

    elif 31 <= dpd <= 45:
        return "31-45"

    elif 46 <= dpd <= 59:
        return "46-59"

    elif dpd >= 60:
        return "60+"

    return np.nan


df["dpd_stage"] = df["days_past_due"].apply(dpd_stage)

stage_order = [
    "01-07",
    "08-15",
    "16-30",
    "31-45",
    "46-59",
    "60+"
]


# somente DPD válido
x = df[
    df["dpd_stage"].notna()
    & df["month_label"].notna()
].copy()


# ============================================================
# 3. CUSTOMER × MONTH × DPD
#
# Esta é a tabela correta para calcular mediana,
# percentis e distribuição de pressão.
# ============================================================

customer_month_dpd = (
    x
    .groupby(
        ["month_label", "dpd_stage", "customer_id"],
        observed=False
    )
    .size()
    .rename("messages")
    .reset_index()
)


# ============================================================
# 4. MONTH × DPD SUMMARY
# ============================================================

monthly_dpd = (
    customer_month_dpd
    .groupby(
        ["month_label", "dpd_stage"],
        observed=False
    )
    .agg(
        customers=("customer_id", "nunique"),
        messages=("messages", "sum"),

        avg_messages_per_customer=("messages", "mean"),
        median_messages_per_customer=("messages", "median"),

        p75_messages_per_customer=(
            "messages",
            lambda s: s.quantile(0.75)
        ),

        p90_messages_per_customer=(
            "messages",
            lambda s: s.quantile(0.90)
        ),

        max_messages_per_customer=("messages", "max")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. SHARE OF MONTHLY MESSAGES
# ------------------------------------------------------------

monthly_dpd["share_messages_pct"] = (
    monthly_dpd["messages"]
    /
    monthly_dpd
        .groupby("month_label")["messages"]
        .transform("sum")
    * 100
)


# ------------------------------------------------------------
# 6. ORDER
# ------------------------------------------------------------

month_order = ["Jun", "Jul", "Aug"]

monthly_dpd["month_label"] = pd.Categorical(
    monthly_dpd["month_label"],
    categories=month_order,
    ordered=True
)

monthly_dpd["dpd_stage"] = pd.Categorical(
    monthly_dpd["dpd_stage"],
    categories=stage_order,
    ordered=True
)

monthly_dpd = (
    monthly_dpd
    .sort_values(
        ["month_label", "dpd_stage"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 7. MONTHLY TABLE
# ============================================================

print("=" * 120)
print("CONTACT EFFORT BY DPD — MONTHLY")
print("=" * 120)

display(
    monthly_dpd[
        [
            "month_label",
            "dpd_stage",
            "customers",
            "messages",
            "share_messages_pct",
            "avg_messages_per_customer",
            "median_messages_per_customer",
            "p75_messages_per_customer",
            "p90_messages_per_customer",
            "max_messages_per_customer"
        ]
    ].style.format(
        {
            "customers": "{:,.0f}",
            "messages": "{:,.0f}",
            "share_messages_pct": "{:.2f}%",
            "avg_messages_per_customer": "{:.2f}",
            "median_messages_per_customer": "{:.1f}",
            "p75_messages_per_customer": "{:.1f}",
            "p90_messages_per_customer": "{:.1f}",
            "max_messages_per_customer": "{:.0f}"
        }
    )
)


# ============================================================
# 8. FULL PERIOD — CUSTOMER × DPD
#
# Aqui o mesmo cliente é contado UMA vez dentro de cada
# DPD stage, independentemente do mês.
# ============================================================

customer_period_dpd = (
    x
    .groupby(
        ["dpd_stage", "customer_id"],
        observed=False
    )
    .size()
    .rename("messages")
    .reset_index()
)


period_dpd = (
    customer_period_dpd
    .groupby(
        "dpd_stage",
        observed=False
    )
    .agg(
        customers=("customer_id", "nunique"),
        messages=("messages", "sum"),

        avg_messages_per_customer=("messages", "mean"),
        median_messages_per_customer=("messages", "median"),

        p75_messages_per_customer=(
            "messages",
            lambda s: s.quantile(0.75)
        ),

        p90_messages_per_customer=(
            "messages",
            lambda s: s.quantile(0.90)
        ),

        max_messages_per_customer=("messages", "max")
    )
    .reset_index()
)


period_dpd["share_messages_pct"] = (
    period_dpd["messages"]
    /
    period_dpd["messages"].sum()
    * 100
)


period_dpd["dpd_stage"] = pd.Categorical(
    period_dpd["dpd_stage"],
    categories=stage_order,
    ordered=True
)

period_dpd = (
    period_dpd
    .sort_values("dpd_stage")
    .reset_index(drop=True)
)


print("\n" + "=" * 120)
print("CONTACT EFFORT BY DPD — FULL PERIOD")
print("=" * 120)

display(
    period_dpd.style.format(
        {
            "customers": "{:,.0f}",
            "messages": "{:,.0f}",
            "share_messages_pct": "{:.2f}%",
            "avg_messages_per_customer": "{:.2f}",
            "median_messages_per_customer": "{:.1f}",
            "p75_messages_per_customer": "{:.1f}",
            "p90_messages_per_customer": "{:.1f}",
            "max_messages_per_customer": "{:.0f}"
        }
    )
)


# ============================================================
# 9. PIVOT — MESSAGE SHARE BY DPD AND MONTH
# ============================================================

message_share_pivot = (
    monthly_dpd
    .pivot(
        index="dpd_stage",
        columns="month_label",
        values="share_messages_pct"
    )
    .reindex(stage_order)
)


print("\n" + "=" * 120)
print("SHARE OF MONTHLY MESSAGES BY DPD")
print("=" * 120)

display(
    message_share_pivot.style.format("{:.2f}%")
)


# ============================================================
# 10. PIVOT — AVG MESSAGES / CUSTOMER
# ============================================================

avg_pivot = (
    monthly_dpd
    .pivot(
        index="dpd_stage",
        columns="month_label",
        values="avg_messages_per_customer"
    )
    .reindex(stage_order)
)


print("\n" + "=" * 120)
print("AVG MESSAGES PER CUSTOMER — MONTH × DPD")
print("=" * 120)

display(
    avg_pivot.style.format("{:.2f}")
)


# ============================================================
# 11. PIVOT — MEDIAN MESSAGES / CUSTOMER
# ============================================================

median_pivot = (
    monthly_dpd
    .pivot(
        index="dpd_stage",
        columns="month_label",
        values="median_messages_per_customer"
    )
    .reindex(stage_order)
)


print("\n" + "=" * 120)
print("MEDIAN MESSAGES PER CUSTOMER — MONTH × DPD")
print("=" * 120)

display(
    median_pivot.style.format("{:.1f}")
)


# ============================================================
# 12. DPD <=30 — FULL PERIOD
#
# Agora queremos especificamente:
# quantos clientes passaram por <=30
# quantas mensagens foram enviadas nessa janela
# quantas mensagens por cliente nessa janela inteira
# ============================================================

early = x[
    x["days_past_due"].between(1, 30)
].copy()


early_customer = (
    early
    .groupby("customer_id")
    .size()
    .rename("messages_dpd_le_30")
    .reset_index()
)


early_summary = pd.DataFrame(
    {
        "metric": [
            "Unique customers DPD <=30",
            "Messages DPD <=30",
            "Share total messages",
            "Average messages/customer",
            "Median messages/customer",
            "P75 messages/customer",
            "P90 messages/customer",
            "Maximum messages/customer"
        ],

        "value": [
            early["customer_id"].nunique(),
            len(early),

            len(early) / len(x) * 100,

            early_customer[
                "messages_dpd_le_30"
            ].mean(),

            early_customer[
                "messages_dpd_le_30"
            ].median(),

            early_customer[
                "messages_dpd_le_30"
            ].quantile(0.75),

            early_customer[
                "messages_dpd_le_30"
            ].quantile(0.90),

            early_customer[
                "messages_dpd_le_30"
            ].max()
        ]
    }
)


print("\n" + "=" * 120)
print("DPD <=30 — CONTACT EFFORT, FULL PERIOD")
print("=" * 120)

display(early_summary)


# ============================================================
# 13. DPD <=30 — MONTHLY
#
# Muito importante para September planning:
# quantos contatos um cliente consome por mês
# enquanto ainda está na janela <=30.
# ============================================================

early_month_customer = (
    early
    .groupby(
        ["month_label", "customer_id"],
        observed=False
    )
    .size()
    .rename("messages")
    .reset_index()
)


early_monthly_summary = (
    early_month_customer
    .groupby(
        "month_label",
        observed=False
    )
    .agg(
        customers=("customer_id", "nunique"),
        messages=("messages", "sum"),

        avg_messages_per_customer=("messages", "mean"),

        median_messages_per_customer=(
            "messages",
            "median"
        ),

        p75_messages_per_customer=(
            "messages",
            lambda s: s.quantile(0.75)
        ),

        p90_messages_per_customer=(
            "messages",
            lambda s: s.quantile(0.90)
        ),

        max_messages_per_customer=(
            "messages",
            "max"
        )
    )
    .reset_index()
)


print("\n" + "=" * 120)
print("DPD <=30 — CONTACT EFFORT BY MONTH")
print("=" * 120)

display(
    early_monthly_summary.style.format(
        {
            "customers": "{:,.0f}",
            "messages": "{:,.0f}",
            "avg_messages_per_customer": "{:.2f}",
            "median_messages_per_customer": "{:.1f}",
            "p75_messages_per_customer": "{:.1f}",
            "p90_messages_per_customer": "{:.1f}",
            "max_messages_per_customer": "{:.0f}"
        }
    )
)


# ============================================================
# 14. JUL–AUG ONLY — DPD <=30
#
# Melhor benchmark operacional para setembro,
# pois Jun sofre left truncation.
# ============================================================

early_ja = early[
    early["month_label"].isin(["Jul", "Aug"])
].copy()


early_ja_customer = (
    early_ja
    .groupby("customer_id")
    .size()
    .rename("messages")
    .reset_index()
)


print("\n" + "=" * 120)
print("DPD <=30 — JUL–AUG COMPARABLE PERIOD")
print("=" * 120)

print(
    f"Unique customers         : "
    f"{early_ja['customer_id'].nunique():,.0f}"
)

print(
    f"Messages                 : "
    f"{len(early_ja):,.0f}"
)

print(
    f"Average msgs/customer    : "
    f"{early_ja_customer['messages'].mean():.2f}"
)

print(
    f"Median msgs/customer     : "
    f"{early_ja_customer['messages'].median():.1f}"
)

print(
    f"P75 msgs/customer        : "
    f"{early_ja_customer['messages'].quantile(.75):.1f}"
)

print(
    f"P90 msgs/customer        : "
    f"{early_ja_customer['messages'].quantile(.90):.1f}"
)

print(
    f"Maximum msgs/customer    : "
    f"{early_ja_customer['messages'].max():.0f}"
)


# ============================================================
# 15. RECONCILIATION
# ============================================================

print("\n" + "=" * 120)
print("RECONCILIATION")
print("=" * 120)

print(
    f"Messages with valid DPD : "
    f"{len(x):,.0f}"
)

print(
    f"Messages in source      : "
    f"{len(df):,.0f}"
)

print(
    f"Unique customers        : "
    f"{x['customer_id'].nunique():,.0f}"
)

CONTACT EFFORT BY DPD — MONTHLY


,month_label,dpd_stage,customers,messages,share_messages_pct,avg_messages_per_customer,median_messages_per_customer,p75_messages_per_customer,p90_messages_per_customer,max_messages_per_customer
0,Jun,01-07,"3,492","7,186",49.68%,2.06,2.0,3.0,3.0,7
1,Jun,08-15,"2,331","5,060",34.98%,2.17,2.0,3.0,4.0,6
2,Jun,16-30,"1,140","2,219",15.34%,1.95,2.0,3.0,4.0,8
3,Jul,01-07,"4,208","8,640",29.08%,2.05,2.0,3.0,3.0,6
4,Jul,08-15,"3,789","8,116",27.31%,2.14,2.0,3.0,4.0,6
5,Jul,16-30,"3,774","8,526",28.69%,2.26,2.0,3.0,4.0,8
6,Jul,31-45,"1,955","3,260",10.97%,1.67,1.0,2.0,3.0,8
7,Jul,46-59,760,"1,159",3.90%,1.52,1.0,2.0,3.0,5
8,Jul,60+,15,15,0.05%,1.00,1.0,1.0,1.0,1
9,Aug,01-07,"3,915","7,898",25.29%,2.02,2.0,3.0,3.0,6



CONTACT EFFORT BY DPD — FULL PERIOD


,dpd_stage,customers,messages,avg_messages_per_customer,median_messages_per_customer,p75_messages_per_customer,p90_messages_per_customer,max_messages_per_customer,share_messages_pct
0,01-07,"11,019","23,724",2.15,2.0,3.0,4.0,7,31.46%
1,08-15,"9,046","20,629",2.28,2.0,3.0,4.0,6,27.36%
2,16-30,"7,179","18,616",2.59,2.0,3.0,4.0,8,24.69%
3,31-45,"4,269","7,474",1.75,2.0,2.0,3.0,8,9.91%
4,46-59,"2,804","4,721",1.68,1.0,2.0,3.0,6,6.26%
5,60+,242,242,1.00,1.0,1.0,1.0,1,0.32%



SHARE OF MONTHLY MESSAGES BY DPD


...


DPD <=30 — CONTACT EFFORT, FULL PERIOD


,metric,value
0,Unique customers DPD <=30,"11,723.00"
1,Messages DPD <=30,"62,969.00"
2,Share total messages,83.51
3,Average messages/customer,5.37
4,Median messages/customer,5.00
5,P75 messages/customer,7.00
6,P90 messages/customer,9.00
7,Maximum messages/customer,16.00



DPD <=30 — CONTACT EFFORT BY MONTH


,month_label,customers,messages,avg_messages_per_customer,median_messages_per_customer,p75_messages_per_customer,p90_messages_per_customer,max_messages_per_customer
0,Aug,"6,295","23,222",3.69,3.0,5.0,7.0,13
1,Jul,"6,616","25,282",3.82,3.0,5.0,7.0,15
2,Jun,"3,681","14,465",3.93,4.0,6.0,7.0,14



DPD <=30 — JUL–AUG COMPARABLE PERIOD
Unique customers         : 10,503
Messages                 : 48,504
Average msgs/customer    : 4.62
Median msgs/customer     : 4.0
P75 msgs/customer        : 7.0
P90 msgs/customer        : 8.0
Maximum msgs/customer    : 16

RECONCILIATION
Messages with valid DPD : 75,406
Messages in source      : 75,406
Unique customers        : 11,724


In [44]:
# ============================================================
# DPD > 30 — SHOULD WE SWITCH TO DISCOUNT OFFER?
#
# Objetivo:
# Comparar clientes após cruzarem DPD30 segundo:
#
# 1. Discount offer vs no discount
# 2. No prior payment vs prior partial payment
# 3. Timing da primeira oferta
# 4. Pressure antes da oferta
# 5. Outcome após a oferta
#
# IMPORTANTE:
# análise observacional / descritiva
# NÃO interpretar diferença como causal.
# ============================================================

import pandas as pd
import numpy as np

df = wa.copy()

df["sent_at"] = pd.to_datetime(df["sent_at"])
df = df.sort_values(["customer_id", "sent_at"]).reset_index(drop=True)

# ------------------------------------------------------------
# 0. BASIC FLAGS
# ------------------------------------------------------------

df["is_discount"] = (
    df["template"]
    .astype(str)
    .str.lower()
    .eq("discount_offer")
)

df["has_payment"] = (
    df["amount_paid_brl"].fillna(0) > 0
)

# ============================================================
# 1. POINT-IN-TIME PAYMENT HISTORY
#
# Tudo que aconteceu ANTES da mensagem atual.
# ============================================================

df["prior_payment_count"] = (
    df.groupby("customer_id")["has_payment"]
      .transform(lambda s: s.shift().fillna(False).cumsum())
)

df["prior_amount_paid_brl"] = (
    df.groupby("customer_id")["amount_paid_brl"]
      .transform(
          lambda s: s.fillna(0)
                     .shift()
                     .fillna(0)
                     .cumsum()
      )
)

df["prior_message_count"] = (
    df.groupby("customer_id")
      .cumcount()
)

df["had_prior_payment"] = (
    df["prior_payment_count"] > 0
)

df["payment_history"] = np.where(
    df["had_prior_payment"],
    "Prior payment",
    "No prior payment"
)


# ============================================================
# 2. ONLY MESSAGES AFTER DPD30
# ============================================================

late = df[
    df["days_past_due"] > 30
].copy()

print("=" * 100)
print("DPD >30 POPULATION")
print("=" * 100)

print(
    f"Customers : {late['customer_id'].nunique():,.0f}"
)

print(
    f"Messages  : {len(late):,.0f}"
)

print(
    f"Discount messages : {late['is_discount'].sum():,.0f}"
)


# ============================================================
# 3. FIRST OBSERVATION AFTER DPD30
#
# Estado do cliente quando entra na late collections.
# ============================================================

late_entry = (
    late
    .sort_values(["customer_id", "sent_at"])
    .groupby("customer_id")
    .first()
    .reset_index()
)

print("\n" + "=" * 100)
print("CUSTOMER STATE WHEN FIRST OBSERVED > DPD30")
print("=" * 100)

entry_summary = (
    late_entry
    .groupby("payment_history")
    .agg(
        customers=("customer_id", "nunique"),
        avg_dpd=("days_past_due", "mean"),
        median_dpd=("days_past_due", "median"),
        avg_prior_messages=("prior_message_count", "mean"),
        median_prior_messages=("prior_message_count", "median"),
        avg_prior_paid=("prior_amount_paid_brl", "mean")
    )
)

display(entry_summary)


# ============================================================
# 4. EVER RECEIVED DISCOUNT AFTER DPD30
# ============================================================

customer_strategy = (
    late.groupby("customer_id")
        .agg(
            received_discount=("is_discount", "max"),
            messages_after_30=("customer_id", "size")
        )
        .reset_index()
)

customer_strategy["strategy"] = np.where(
    customer_strategy["received_discount"],
    "Received discount",
    "Never received discount"
)

# attach state at DPD30
customer_strategy = customer_strategy.merge(
    late_entry[
        [
            "customer_id",
            "payment_history",
            "days_past_due",
            "prior_message_count",
            "prior_amount_paid_brl",
            "outstanding_balance_brl"
        ]
    ],
    on="customer_id",
    how="left"
)


# ============================================================
# 5. STRATEGY DISTRIBUTION
# ============================================================

strategy_distribution = (
    customer_strategy
    .groupby(
        ["payment_history", "strategy"]
    )
    .agg(
        customers=("customer_id", "nunique"),
        avg_entry_dpd=("days_past_due", "mean"),
        avg_prior_messages=("prior_message_count", "mean"),
        median_prior_messages=("prior_message_count", "median"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)

strategy_distribution["share_within_history_pct"] = (
    strategy_distribution["customers"]
    /
    strategy_distribution
        .groupby("payment_history")["customers"]
        .transform("sum")
    * 100
)

print("\n" + "=" * 100)
print("WHO RECEIVES DISCOUNT AFTER DPD30?")
print("=" * 100)

display(
    strategy_distribution.style.format(
        {
            "customers": "{:,.0f}",
            "avg_entry_dpd": "{:.1f}",
            "avg_prior_messages": "{:.2f}",
            "median_prior_messages": "{:.1f}",
            "avg_balance": "R$ {:,.2f}",
            "share_within_history_pct": "{:.1f}%"
        }
    )
)


# ============================================================
# 6. FIRST DISCOUNT AFTER DPD30
# ============================================================

first_discount = (
    late[
        late["is_discount"]
    ]
    .sort_values(["customer_id", "sent_at"])
    .groupby("customer_id")
    .first()
    .reset_index()
)

print("\n" + "=" * 100)
print("FIRST DISCOUNT OFFER AFTER DPD30")
print("=" * 100)

print(
    f"Customers receiving discount : "
    f"{first_discount['customer_id'].nunique():,.0f}"
)

print(
    f"Average DPD at first discount : "
    f"{first_discount['days_past_due'].mean():.1f}"
)

print(
    f"Median DPD at first discount  : "
    f"{first_discount['days_past_due'].median():.1f}"
)

print(
    f"Average prior messages        : "
    f"{first_discount['prior_message_count'].mean():.2f}"
)

print(
    f"Median prior messages         : "
    f"{first_discount['prior_message_count'].median():.1f}"
)


# ============================================================
# 7. DPD BUCKET AT FIRST DISCOUNT
# ============================================================

first_discount["discount_dpd_bucket"] = pd.cut(
    first_discount["days_past_due"],
    bins=[30, 45, 59, np.inf],
    labels=[
        "31-45",
        "46-59",
        "60+"
    ]
)

discount_timing = (
    first_discount
    .groupby(
        "discount_dpd_bucket",
        observed=False
    )
    .agg(
        customers=("customer_id", "nunique"),
        avg_prior_messages=("prior_message_count", "mean"),
        median_prior_messages=("prior_message_count", "median")
    )
    .reset_index()
)

discount_timing["share_pct"] = (
    discount_timing["customers"]
    /
    discount_timing["customers"].sum()
    * 100
)

print("\n" + "=" * 100)
print("WHEN IS THE FIRST DISCOUNT OFFERED?")
print("=" * 100)

display(
    discount_timing.style.format(
        {
            "customers": "{:,.0f}",
            "avg_prior_messages": "{:.2f}",
            "median_prior_messages": "{:.1f}",
            "share_pct": "{:.1f}%"
        }
    )
)


# ============================================================
# 8. OUTCOME OF FIRST DISCOUNT
#
# paid_within_72h já é nosso label pós-ação.
# amount_paid_brl = recovery atribuído.
# ============================================================

first_discount["paid_72h"] = (
    first_discount["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

discount_outcome = (
    first_discount
    .groupby(
        "payment_history"
    )
    .agg(
        customers=("customer_id", "nunique"),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate_72h=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "amount_paid_brl",
            "sum"
        ),

        avg_recovery_per_customer=(
            "amount_paid_brl",
            "mean"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        )
    )
    .reset_index()
)

discount_outcome["payment_rate_72h"] *= 100

discount_outcome["recovery_to_balance_pct"] = (
    discount_outcome["recovery_brl"]
    /
    (
        discount_outcome["avg_balance"]
        *
        discount_outcome["customers"]
    )
    * 100
)

print("\n" + "=" * 100)
print("FIRST DISCOUNT OUTCOME — BY PRIOR PAYMENT HISTORY")
print("=" * 100)

display(
    discount_outcome.style.format(
        {
            "customers": "{:,.0f}",
            "payment_events": "{:,.0f}",
            "payment_rate_72h": "{:.2f}%",
            "recovery_brl": "R$ {:,.2f}",
            "avg_recovery_per_customer": "R$ {:,.2f}",
            "avg_balance": "R$ {:,.2f}",
            "recovery_to_balance_pct": "{:.2f}%"
        }
    )
)


# ============================================================
# 9. RESPONSE BY DPD OF FIRST DISCOUNT
# ============================================================

discount_by_dpd = (
    first_discount
    .groupby(
        "discount_dpd_bucket",
        observed=False
    )
    .agg(
        customers=("customer_id", "nunique"),

        payment_events=("paid_72h", "sum"),

        payment_rate_72h=("paid_72h", "mean"),

        recovery_brl=("amount_paid_brl", "sum"),

        avg_recovery_per_customer=(
            "amount_paid_brl",
            "mean"
        ),

        avg_prior_messages=(
            "prior_message_count",
            "mean"
        )
    )
    .reset_index()
)

discount_by_dpd["payment_rate_72h"] *= 100

print("\n" + "=" * 100)
print("FIRST DISCOUNT OUTCOME — BY DPD")
print("=" * 100)

display(
    discount_by_dpd.style.format(
        {
            "customers": "{:,.0f}",
            "payment_events": "{:,.0f}",
            "payment_rate_72h": "{:.2f}%",
            "recovery_brl": "R$ {:,.2f}",
            "avg_recovery_per_customer": "R$ {:,.2f}",
            "avg_prior_messages": "{:.2f}"
        }
    )
)


# ============================================================
# 10. RESPONSE BY PRIOR CONTACT PRESSURE
# ============================================================

first_discount["prior_pressure_bucket"] = pd.cut(
    first_discount["prior_message_count"],
    bins=[-1, 3, 5, 7, 10, np.inf],
    labels=[
        "0-3",
        "4-5",
        "6-7",
        "8-10",
        "11+"
    ]
)

discount_by_pressure = (
    first_discount
    .groupby(
        "prior_pressure_bucket",
        observed=False
    )
    .agg(
        customers=("customer_id", "nunique"),

        avg_dpd=("days_past_due", "mean"),

        payment_events=("paid_72h", "sum"),

        payment_rate_72h=("paid_72h", "mean"),

        recovery_brl=("amount_paid_brl", "sum"),

        avg_recovery_per_customer=(
            "amount_paid_brl",
            "mean"
        )
    )
    .reset_index()
)

discount_by_pressure["payment_rate_72h"] *= 100

print("\n" + "=" * 100)
print("FIRST DISCOUNT OUTCOME — PRIOR CONTACT PRESSURE")
print("=" * 100)

display(
    discount_by_pressure.style.format(
        {
            "customers": "{:,.0f}",
            "avg_dpd": "{:.1f}",
            "payment_events": "{:,.0f}",
            "payment_rate_72h": "{:.2f}%",
            "recovery_brl": "R$ {:,.2f}",
            "avg_recovery_per_customer": "R$ {:,.2f}"
        }
    )
)


# ============================================================
# 11. REPEATED DISCOUNT OFFERS
#
# Queremos descobrir se discount também vira "insistência".
# ============================================================

discount_msgs = (
    late[
        late["is_discount"]
    ]
    .copy()
)

discount_msgs["discount_number"] = (
    discount_msgs
    .groupby("customer_id")
    .cumcount()
    + 1
)

discount_sequence = (
    discount_msgs
    .groupby("discount_number")
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),

        payment_events=(
            "paid_within_72h",
            "sum"
        ),

        payment_rate_72h=(
            "paid_within_72h",
            "mean"
        ),

        recovery_brl=(
            "amount_paid_brl",
            "sum"
        ),

        avg_recovery_per_message=(
            "amount_paid_brl",
            "mean"
        )
    )
    .reset_index()
)

discount_sequence["payment_rate_72h"] *= 100

print("\n" + "=" * 100)
print("DISCOUNT OFFER SEQUENCE")
print("=" * 100)

display(
    discount_sequence.style.format(
        {
            "messages": "{:,.0f}",
            "customers": "{:,.0f}",
            "payment_events": "{:,.0f}",
            "payment_rate_72h": "{:.2f}%",
            "recovery_brl": "R$ {:,.2f}",
            "avg_recovery_per_message": "R$ {:,.2f}"
        }
    )
)

DPD >30 POPULATION
Customers : 4,884
Messages  : 12,437
Discount messages : 6,218

CUSTOMER STATE WHEN FIRST OBSERVED > DPD30


,customers,avg_dpd,median_dpd,avg_prior_messages,median_prior_messages,avg_prior_paid
payment_history,,,,,,
No prior payment,4130,37.59,36.00,7.09,7.00,0.00
Prior payment,754,38.27,36.00,6.79,7.00,435.43



WHO RECEIVES DISCOUNT AFTER DPD30?


,payment_history,strategy,customers,avg_entry_dpd,avg_prior_messages,median_prior_messages,avg_balance,share_within_history_pct
0,No prior payment,Never received discount,983,38.4,7.12,7.0,R$ 858.39,23.8%
1,No prior payment,Received discount,"3,147",37.3,7.08,7.0,R$ 864.77,76.2%
2,Prior payment,Never received discount,221,39.1,6.77,6.0,R$ 415.71,29.3%
3,Prior payment,Received discount,533,37.9,6.80,7.0,R$ 374.60,70.7%



FIRST DISCOUNT OFFER AFTER DPD30
Customers receiving discount : 3,680
Average DPD at first discount : 40.3
Median DPD at first discount  : 38.5
Average prior messages        : 7.50
Median prior messages         : 7.0

WHEN IS THE FIRST DISCOUNT OFFERED?


...

,discount_dpd_bucket,customers,payment_events,payment_rate_72h,recovery_brl,avg_recovery_per_customer,avg_prior_messages
0,31-45,"2,802",250,8.92%,"R$ 133,358.18",R$ 47.59,7.32
1,46-59,854,57,6.67%,"R$ 26,085.11",R$ 30.54,8.09
2,60+,24,1,4.17%,R$ 739.72,R$ 30.82,7.75



FIRST DISCOUNT OUTCOME — PRIOR CONTACT PRESSURE


,prior_pressure_bucket,customers,avg_dpd,payment_events,payment_rate_72h,recovery_brl,avg_recovery_per_customer
0,0-3,128,37.9,17,13.28%,"R$ 8,658.26",R$ 67.64
1,4-5,650,39.4,61,9.38%,"R$ 28,968.66",R$ 44.57
2,6-7,"1,101",39.2,99,8.99%,"R$ 52,794.27",R$ 47.95
3,8-10,"1,403",41.1,107,7.63%,"R$ 55,326.66",R$ 39.43
4,11+,398,42.7,24,6.03%,"R$ 14,435.16",R$ 36.27



DISCOUNT OFFER SEQUENCE


,discount_number,messages,customers,payment_events,payment_rate_72h,recovery_brl,avg_recovery_per_message
0,1,"3,680","3,680",308,8.37%,"R$ 160,183.01",R$ 43.53
1,2,"1,689","1,689",116,6.87%,"R$ 59,499.14",R$ 35.23
2,3,623,623,32,5.14%,"R$ 18,557.42",R$ 29.79
3,4,172,172,6,3.49%,"R$ 4,343.72",R$ 25.25
4,5,43,43,3,6.98%,R$ 999.83,R$ 23.25
5,6,9,9,0,0.00%,R$ 0.00,R$ 0.00
6,7,2,2,0,0.00%,R$ 0.00,R$ 0.00


In [45]:
# ============================================================
# DPD >30 — PRIOR PAYERS:
# DISCOUNT OFFER vs NO DISCOUNT OFFER
#
# POPULATION:
# Customers who:
#   1. had >=1 payment while DPD <=30
#   2. were subsequently observed at DPD >30
#
# COMPARISON AFTER DPD30:
#   A. Received discount_offer
#   B. Never received discount_offer
#
# IMPORTANT:
# Observational comparison — NOT causal.
# ============================================================

import pandas as pd
import numpy as np

df = wa.copy()

df["sent_at"] = pd.to_datetime(df["sent_at"])

df = (
    df
    .sort_values(["customer_id", "sent_at"])
    .reset_index(drop=True)
)

df["amount_paid_brl"] = (
    pd.to_numeric(
        df["amount_paid_brl"],
        errors="coerce"
    )
    .fillna(0)
)

df["is_payment"] = df["amount_paid_brl"] > 0

df["is_discount"] = (
    df["template"]
    .astype(str)
    .str.lower()
    .eq("discount_offer")
)


# ============================================================
# 1. IDENTIFY PAYMENTS WHILE DPD <=30
# ============================================================

pre30_payments = df[
    (df["days_past_due"] <= 30)
    & (df["is_payment"])
].copy()


prior_payer_ids = set(
    pre30_payments["customer_id"].unique()
)

print("=" * 100)
print("CUSTOMERS WITH PAYMENT AT DPD <=30")
print("=" * 100)

print(
    f"Customers : "
    f"{len(prior_payer_ids):,.0f}"
)


# ============================================================
# 2. IDENTIFY CUSTOMERS OBSERVED AFTER DPD30
# ============================================================

post30 = df[
    df["days_past_due"] > 30
].copy()

post30_ids = set(
    post30["customer_id"].unique()
)


# ============================================================
# 3. FINAL COHORT
#
# Payment <=30 AND subsequently observed >30
# ============================================================

cohort_ids = (
    prior_payer_ids
    .intersection(post30_ids)
)

cohort = post30[
    post30["customer_id"].isin(cohort_ids)
].copy()


print("\n" + "=" * 100)
print("FINAL PRIOR-PAYER COHORT OBSERVED AFTER DPD30")
print("=" * 100)

print(
    f"Customers : "
    f"{cohort['customer_id'].nunique():,.0f}"
)

print(
    f"Messages after DPD30 : "
    f"{len(cohort):,.0f}"
)


# ============================================================
# 4. STRATEGY AFTER DPD30
# ============================================================

strategy = (
    cohort
    .groupby("customer_id")
    .agg(
        received_discount_after30=(
            "is_discount",
            "max"
        )
    )
    .reset_index()
)

strategy["strategy"] = np.where(
    strategy["received_discount_after30"],
    "Discount after DPD30",
    "No discount after DPD30"
)


print("\n" + "=" * 100)
print("STRATEGY DISTRIBUTION")
print("=" * 100)

strategy_dist = (
    strategy
    .groupby("strategy")
    .agg(
        customers=("customer_id", "nunique")
    )
    .reset_index()
)

strategy_dist["share_pct"] = (
    strategy_dist["customers"]
    /
    strategy_dist["customers"].sum()
    * 100
)

display(
    strategy_dist.style.format(
        {
            "customers": "{:,.0f}",
            "share_pct": "{:.1f}%"
        }
    )
)


# ============================================================
# 5. PRE-30 PAYMENT HISTORY
#
# Important:
# Check whether groups were already different BEFORE discount.
# ============================================================

pre30_history = (
    pre30_payments[
        pre30_payments["customer_id"].isin(cohort_ids)
    ]
    .groupby("customer_id")
    .agg(
        payments_pre30=("is_payment", "sum"),
        amount_paid_pre30=("amount_paid_brl", "sum"),
        first_payment_dpd=("days_past_due", "min"),
        last_payment_dpd=("days_past_due", "max")
    )
    .reset_index()
)


# all messages <=30
pre30_messages = (
    df[
        (df["customer_id"].isin(cohort_ids))
        & (df["days_past_due"] <= 30)
    ]
    .groupby("customer_id")
    .agg(
        messages_pre30=("customer_id", "size")
    )
    .reset_index()
)


pre_state = (
    strategy
    .merge(
        pre30_history,
        on="customer_id",
        how="left"
    )
    .merge(
        pre30_messages,
        on="customer_id",
        how="left"
    )
)


pre_comparison = (
    pre_state
    .groupby("strategy")
    .agg(
        customers=("customer_id", "nunique"),

        avg_payments_pre30=(
            "payments_pre30",
            "mean"
        ),

        median_payments_pre30=(
            "payments_pre30",
            "median"
        ),

        avg_amount_paid_pre30=(
            "amount_paid_pre30",
            "mean"
        ),

        median_amount_paid_pre30=(
            "amount_paid_pre30",
            "median"
        ),

        avg_messages_pre30=(
            "messages_pre30",
            "mean"
        ),

        median_messages_pre30=(
            "messages_pre30",
            "median"
        ),

        avg_last_payment_dpd=(
            "last_payment_dpd",
            "mean"
        )
    )
    .reset_index()
)


print("\n" + "=" * 100)
print("PRE-DPD30 BEHAVIOR — BEFORE STRATEGY")
print("=" * 100)

display(
    pre_comparison.style.format(
        {
            "customers": "{:,.0f}",
            "avg_payments_pre30": "{:.2f}",
            "median_payments_pre30": "{:.1f}",
            "avg_amount_paid_pre30": "R$ {:,.2f}",
            "median_amount_paid_pre30": "R$ {:,.2f}",
            "avg_messages_pre30": "{:.2f}",
            "median_messages_pre30": "{:.1f}",
            "avg_last_payment_dpd": "{:.1f}"
        }
    )
)


# ============================================================
# 6. POST-30 OUTCOMES
# ============================================================

post_customer = (
    cohort
    .groupby("customer_id")
    .agg(
        messages_after30=("customer_id", "size"),

        payments_after30=("is_payment", "sum"),

        amount_paid_after30=(
            "amount_paid_brl",
            "sum"
        ),

        first_dpd_after30=(
            "days_past_due",
            "min"
        ),

        last_dpd_observed=(
            "days_past_due",
            "max"
        )
    )
    .reset_index()
)


post_customer["paid_after30"] = (
    post_customer["payments_after30"] > 0
)


post_customer = post_customer.merge(
    strategy[
        [
            "customer_id",
            "strategy"
        ]
    ],
    on="customer_id",
    how="left"
)


# ============================================================
# 7. MAIN COMPARISON
# ============================================================

comparison = (
    post_customer
    .groupby("strategy")
    .agg(
        customers=("customer_id", "nunique"),

        customers_paid_after30=(
            "paid_after30",
            "sum"
        ),

        payment_rate_after30=(
            "paid_after30",
            "mean"
        ),

        payment_events_after30=(
            "payments_after30",
            "sum"
        ),

        total_recovery_after30=(
            "amount_paid_after30",
            "sum"
        ),

        avg_recovery_per_customer=(
            "amount_paid_after30",
            "mean"
        ),

        median_recovery_per_customer=(
            "amount_paid_after30",
            "median"
        ),

        avg_messages_after30=(
            "messages_after30",
            "mean"
        ),

        median_messages_after30=(
            "messages_after30",
            "median"
        ),

        avg_last_dpd=(
            "last_dpd_observed",
            "mean"
        )
    )
    .reset_index()
)


comparison["payment_rate_after30"] *= 100


print("\n" + "=" * 100)
print("POST-DPD30 BEHAVIOR — DISCOUNT vs NO DISCOUNT")
print("=" * 100)

display(
    comparison.style.format(
        {
            "customers": "{:,.0f}",
            "customers_paid_after30": "{:,.0f}",
            "payment_rate_after30": "{:.2f}%",
            "payment_events_after30": "{:,.0f}",
            "total_recovery_after30": "R$ {:,.2f}",
            "avg_recovery_per_customer": "R$ {:,.2f}",
            "median_recovery_per_customer": "R$ {:,.2f}",
            "avg_messages_after30": "{:.2f}",
            "median_messages_after30": "{:.1f}",
            "avg_last_dpd": "{:.1f}"
        }
    )
)


# ============================================================
# 8. RECOVERY AMONG PAYERS ONLY
#
# Separates:
# "probability of paying"
# from
# "how much they pay conditional on paying"
# ============================================================

payers_only = post_customer[
    post_customer["paid_after30"]
].copy()


payer_comparison = (
    payers_only
    .groupby("strategy")
    .agg(
        paying_customers=("customer_id", "nunique"),

        avg_recovery_if_paid=(
            "amount_paid_after30",
            "mean"
        ),

        median_recovery_if_paid=(
            "amount_paid_after30",
            "median"
        ),

        avg_payment_events_if_paid=(
            "payments_after30",
            "mean"
        )
    )
    .reset_index()
)


print("\n" + "=" * 100)
print("POST-DPD30 — AMONG CUSTOMERS WHO PAID")
print("=" * 100)

display(
    payer_comparison.style.format(
        {
            "paying_customers": "{:,.0f}",
            "avg_recovery_if_paid": "R$ {:,.2f}",
            "median_recovery_if_paid": "R$ {:,.2f}",
            "avg_payment_events_if_paid": "{:.2f}"
        }
    )
)


# ============================================================
# 9. MESSAGE EFFICIENCY
# ============================================================

efficiency = (
    post_customer
    .groupby("strategy")
    .agg(
        customers=("customer_id", "nunique"),
        messages=("messages_after30", "sum"),
        recovery=("amount_paid_after30", "sum")
    )
    .reset_index()
)


efficiency["recovery_per_message"] = (
    efficiency["recovery"]
    /
    efficiency["messages"]
)

efficiency["cost_brl"] = (
    efficiency["messages"] * 1.0
)

efficiency["net_recovery_after_message_cost"] = (
    efficiency["recovery"]
    -
    efficiency["cost_brl"]
)


print("\n" + "=" * 100)
print("POST-DPD30 — MESSAGE EFFICIENCY")
print("=" * 100)

display(
    efficiency.style.format(
        {
            "customers": "{:,.0f}",
            "messages": "{:,.0f}",
            "recovery": "R$ {:,.2f}",
            "recovery_per_message": "R$ {:,.2f}",
            "cost_brl": "R$ {:,.2f}",
            "net_recovery_after_message_cost": "R$ {:,.2f}"
        }
    )
)

CUSTOMERS WITH PAYMENT AT DPD <=30
Customers : 4,355

FINAL PRIOR-PAYER COHORT OBSERVED AFTER DPD30
Customers : 754
Messages after DPD30 : 1,802

STRATEGY DISTRIBUTION


,strategy,customers,share_pct
0,Discount after DPD30,533,70.7%
1,No discount after DPD30,221,29.3%



PRE-DPD30 BEHAVIOR — BEFORE STRATEGY


,strategy,customers,avg_payments_pre30,median_payments_pre30,avg_amount_paid_pre30,median_amount_paid_pre30,avg_messages_pre30,median_messages_pre30,avg_last_payment_dpd
0,Discount after DPD30,533,1.06,1.0,R$ 411.05,R$ 341.05,6.80,7.0,12.7
1,No discount after DPD30,221,1.18,1.0,R$ 494.22,R$ 414.64,6.77,6.0,13.5



POST-DPD30 BEHAVIOR — DISCOUNT vs NO DISCOUNT


,strategy,customers,customers_paid_after30,payment_rate_after30,payment_events_after30,total_recovery_after30,avg_recovery_per_customer,median_recovery_per_customer,avg_messages_after30,median_messages_after30,avg_last_dpd
0,Discount after DPD30,533,161,30.21%,170,"R$ 40,726.90",R$ 76.41,R$ 0.00,2.65,2.0,47.7
1,No discount after DPD30,221,51,23.08%,53,"R$ 17,467.49",R$ 79.04,R$ 0.00,1.75,1.0,44.6



POST-DPD30 — AMONG CUSTOMERS WHO PAID


,strategy,paying_customers,avg_recovery_if_paid,median_recovery_if_paid,avg_payment_events_if_paid
0,Discount after DPD30,161,R$ 252.96,R$ 190.75,1.06
1,No discount after DPD30,51,R$ 342.50,R$ 270.11,1.04



POST-DPD30 — MESSAGE EFFICIENCY


,strategy,customers,messages,recovery,recovery_per_message,cost_brl,net_recovery_after_message_cost
0,Discount after DPD30,533,"1,415","R$ 40,726.90",R$ 28.78,"R$ 1,415.00","R$ 39,311.90"
1,No discount after DPD30,221,387,"R$ 17,467.49",R$ 45.14,R$ 387.00,"R$ 17,080.49"


In [46]:
# ============================================================
# PRIOR PAYERS — DPD >30
# DISCOUNT vs NO DISCOUNT
#
# FINAL ECONOMIC COMPARISON
#
# Denominator:
# outstanding balance at FIRST observation after DPD30
#
# Outcomes:
# - payment incidence
# - total recovery after DPD30
# - recovery / opening balance
# - recovery per customer
# - remaining balance
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. OPENING STATE AT DPD >30
# ------------------------------------------------------------

opening_post30 = (
    cohort
    .sort_values(["customer_id", "sent_at"])
    .groupby("customer_id")
    .first()
    .reset_index()
    [
        [
            "customer_id",
            "sent_at",
            "days_past_due",
            "outstanding_balance_brl"
        ]
    ]
    .rename(
        columns={
            "sent_at": "entry_date_post30",
            "days_past_due": "entry_dpd_post30",
            "outstanding_balance_brl": "opening_balance_post30"
        }
    )
)


# ------------------------------------------------------------
# 2. CUSTOMER-LEVEL OUTCOMES AFTER DPD30
# ------------------------------------------------------------

customer_outcome = (
    cohort
    .groupby("customer_id")
    .agg(
        messages_after30=("customer_id", "size"),

        payment_events_after30=(
            "is_payment",
            "sum"
        ),

        recovery_after30=(
            "amount_paid_brl",
            "sum"
        ),

        received_discount=(
            "is_discount",
            "max"
        )
    )
    .reset_index()
)


customer_outcome["strategy"] = np.where(
    customer_outcome["received_discount"],
    "Discount after DPD30",
    "No discount after DPD30"
)


customer_outcome["paid_after30"] = (
    customer_outcome["payment_events_after30"] > 0
)


# ------------------------------------------------------------
# 3. MERGE OPENING BALANCE
# ------------------------------------------------------------

analysis = (
    customer_outcome
    .merge(
        opening_post30,
        on="customer_id",
        how="left"
    )
)


# ------------------------------------------------------------
# 4. CUSTOMER-LEVEL RECOVERY RATE
# ------------------------------------------------------------

analysis["recovery_rate_post30"] = np.where(
    analysis["opening_balance_post30"] > 0,

    analysis["recovery_after30"]
    /
    analysis["opening_balance_post30"],

    np.nan
)


# cap ONLY for descriptive customer-level statistics
# do NOT use capped rate for portfolio recovery
analysis["recovery_rate_post30_capped"] = (
    analysis["recovery_rate_post30"]
    .clip(upper=1)
)


# remaining balance approximation
analysis["remaining_balance"] = (
    analysis["opening_balance_post30"]
    -
    analysis["recovery_after30"]
).clip(lower=0)


# ------------------------------------------------------------
# 5. MAIN PORTFOLIO COMPARISON
# ------------------------------------------------------------

portfolio = (
    analysis
    .groupby("strategy")
    .agg(
        customers=("customer_id", "nunique"),

        opening_balance=(
            "opening_balance_post30",
            "sum"
        ),

        customers_paid=(
            "paid_after30",
            "sum"
        ),

        total_recovery=(
            "recovery_after30",
            "sum"
        ),

        remaining_balance=(
            "remaining_balance",
            "sum"
        ),

        messages=(
            "messages_after30",
            "sum"
        )
    )
    .reset_index()
)


portfolio["payment_rate_pct"] = (
    portfolio["customers_paid"]
    /
    portfolio["customers"]
    * 100
)


portfolio["portfolio_recovery_rate_pct"] = (
    portfolio["total_recovery"]
    /
    portfolio["opening_balance"]
    * 100
)


portfolio["recovery_per_customer"] = (
    portfolio["total_recovery"]
    /
    portfolio["customers"]
)


portfolio["opening_balance_per_customer"] = (
    portfolio["opening_balance"]
    /
    portfolio["customers"]
)


portfolio["recovery_per_message"] = (
    portfolio["total_recovery"]
    /
    portfolio["messages"]
)


print("=" * 110)
print("PRIOR PAYERS — ECONOMIC OUTCOME AFTER DPD30")
print("=" * 110)

display(
    portfolio.style.format(
        {
            "customers": "{:,.0f}",

            "opening_balance":
                "R$ {:,.2f}",

            "opening_balance_per_customer":
                "R$ {:,.2f}",

            "customers_paid":
                "{:,.0f}",

            "payment_rate_pct":
                "{:.2f}%",

            "total_recovery":
                "R$ {:,.2f}",

            "portfolio_recovery_rate_pct":
                "{:.2f}%",

            "recovery_per_customer":
                "R$ {:,.2f}",

            "remaining_balance":
                "R$ {:,.2f}",

            "messages":
                "{:,.0f}",

            "recovery_per_message":
                "R$ {:,.2f}"
        }
    )
)


# ============================================================
# 6. CUSTOMER-LEVEL DISTRIBUTION
# ============================================================

customer_stats = (
    analysis
    .groupby("strategy")
    .agg(
        customers=("customer_id", "nunique"),

        avg_opening_balance=(
            "opening_balance_post30",
            "mean"
        ),

        median_opening_balance=(
            "opening_balance_post30",
            "median"
        ),

        avg_recovery=(
            "recovery_after30",
            "mean"
        ),

        median_recovery=(
            "recovery_after30",
            "median"
        ),

        avg_recovery_rate=(
            "recovery_rate_post30_capped",
            "mean"
        ),

        median_recovery_rate=(
            "recovery_rate_post30_capped",
            "median"
        )
    )
    .reset_index()
)


customer_stats["avg_recovery_rate"] *= 100
customer_stats["median_recovery_rate"] *= 100


print("\n" + "=" * 110)
print("CUSTOMER-LEVEL RECOVERY DISTRIBUTION")
print("=" * 110)

display(
    customer_stats.style.format(
        {
            "customers":
                "{:,.0f}",

            "avg_opening_balance":
                "R$ {:,.2f}",

            "median_opening_balance":
                "R$ {:,.2f}",

            "avg_recovery":
                "R$ {:,.2f}",

            "median_recovery":
                "R$ {:,.2f}",

            "avg_recovery_rate":
                "{:.2f}%",

            "median_recovery_rate":
                "{:.2f}%"
        }
    )
)


# ============================================================
# 7. PAYERS ONLY
#
# Conditional recovery rate:
# once the customer pays, how much of opening balance is recovered?
# ============================================================

payer_analysis = analysis[
    analysis["paid_after30"]
].copy()


payer_stats = (
    payer_analysis
    .groupby("strategy")
    .agg(
        paying_customers=(
            "customer_id",
            "nunique"
        ),

        avg_opening_balance=(
            "opening_balance_post30",
            "mean"
        ),

        avg_recovery=(
            "recovery_after30",
            "mean"
        ),

        median_recovery=(
            "recovery_after30",
            "median"
        ),

        avg_recovery_rate=(
            "recovery_rate_post30_capped",
            "mean"
        ),

        median_recovery_rate=(
            "recovery_rate_post30_capped",
            "median"
        )
    )
    .reset_index()
)


payer_stats["avg_recovery_rate"] *= 100
payer_stats["median_recovery_rate"] *= 100


print("\n" + "=" * 110)
print("AMONG PAYERS — % OF OPENING BALANCE RECOVERED")
print("=" * 110)

display(
    payer_stats.style.format(
        {
            "paying_customers":
                "{:,.0f}",

            "avg_opening_balance":
                "R$ {:,.2f}",

            "avg_recovery":
                "R$ {:,.2f}",

            "median_recovery":
                "R$ {:,.2f}",

            "avg_recovery_rate":
                "{:.2f}%",

            "median_recovery_rate":
                "{:.2f}%"
        }
    )
)

PRIOR PAYERS — ECONOMIC OUTCOME AFTER DPD30


,strategy,customers,opening_balance,customers_paid,total_recovery,remaining_balance,messages,payment_rate_pct,portfolio_recovery_rate_pct,recovery_per_customer,opening_balance_per_customer,recovery_per_message
0,Discount after DPD30,533,"R$ 199,662.66",161,"R$ 40,726.90","R$ 158,935.76","1,415",30.21%,20.40%,R$ 76.41,R$ 374.60,R$ 28.78
1,No discount after DPD30,221,"R$ 91,871.97",51,"R$ 17,467.49","R$ 74,404.49",387,23.08%,19.01%,R$ 79.04,R$ 415.71,R$ 45.14



CUSTOMER-LEVEL RECOVERY DISTRIBUTION


,strategy,customers,avg_opening_balance,median_opening_balance,avg_recovery,median_recovery,avg_recovery_rate,median_recovery_rate
0,Discount after DPD30,533,R$ 374.60,R$ 300.00,R$ 76.41,R$ 0.00,23.53%,0.00%
1,No discount after DPD30,221,R$ 415.71,R$ 331.81,R$ 79.04,R$ 0.00,18.86%,0.00%



AMONG PAYERS — % OF OPENING BALANCE RECOVERED


,strategy,paying_customers,avg_opening_balance,avg_recovery,median_recovery,avg_recovery_rate,median_recovery_rate
0,Discount after DPD30,161,R$ 324.58,R$ 252.96,R$ 190.75,77.90%,85.00%
1,No discount after DPD30,51,R$ 411.14,R$ 342.50,R$ 270.11,81.72%,100.00%
